# R2AI Kaggle Inference Notebook (Multi-GPU)
Notebook này được thiết kế để chạy pipeline RAG offline trên Kaggle sử dụng SQLite DB và Local FAISS Index.
Phiên bản này sử dụng Multiprocessing để tận dụng tối đa 2 GPUs (ví dụ T4x2) giúp tốc độ sinh câu trả lời nhanh gấp đôi.

In [13]:
!pip install -q transformers sentence-transformers faiss-gpu underthesea accelerate bitsandbytes

In [14]:
import os
import sys
import json
import time
import shutil

# 1. Khai báo các đường dẫn trên Kaggle
# ĐIỀU CHỈNH: Tên dataset của bạn trên Kaggle (ví dụ: r2ai-project)
KAGGLE_DATASET_PATH = "/kaggle/input/datasets/triuinhhi/r2aiiii"

# Thư mục làm việc trên Kaggle
WORKING_DIR = "/kaggle/working"
PROJECT_ROOT = os.path.join(WORKING_DIR, "R2AI")

# 2. Copy toàn bộ code và database từ thư mục Input sang thư mục Working (để có thể ghi file)
if not os.path.exists(PROJECT_ROOT):
    print("Copying project from input to working directory...")
    shutil.copytree(KAGGLE_DATASET_PATH, PROJECT_ROOT)
    print("Copy completed!")

# 3. Thêm project root vào thư mục path để import
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print(f"Project root added to path: {PROJECT_ROOT}")

Project root added to path: /kaggle/working/R2AI


In [15]:
# 4. Cấu hình biến môi trường để chạy Offline hoàn toàn
os.environ["HF_HUB_OFFLINE"] = "0"  # Tắt Offline mode để HuggingFace tải weights nếu chưa có trên Kaggle

# Tắt cảnh báo Tokenizer
import logging
import transformers
transformers.logging.set_verbosity_error()
logging.getLogger("transformers.tokenization_utils_base").setLevel(logging.ERROR)

In [17]:

# ==========================================
# File: prompts/prompt_templates.py
# ==========================================

# Prompt templates for R2AI generator and rewriter

SYSTEM_PROMPT = (
    "Bạn là một chuyên gia phân tích pháp lý cao cấp. Nhiệm vụ của bạn là trích dẫn thông tin từ ngữ cảnh (Context) được cung cấp để trả lời câu hỏi một cách chuẩn xác, khách quan.\n\n"
    "HƯỚNG DẪN TƯ DUY VÀ KHỬ LỖI:\n"
    "1. Trả lời trực tiếp: Đưa ra câu trả lời kết luận ngắn gọn, trực diện trong 1-2 câu. (Ví dụ: Nếu hỏi Có/Không thì khẳng định Có/Không kèm lý do ngắn; nếu hỏi 'Mức nào/Điều kiện gì/Bao lâu' thì chỉ ra ngay con số hoặc điều kiện cốt lõi nhất, KHÔNG ghi chung chung là 'Khẳng định Có').\n"
    "2. TUYỆT ĐỐI KHÔNG LẶP Ý: Không copy lại các câu chữ, nội dung đã viết ở phần trước xuống phần sau. Mỗi thông tin chỉ xuất hiện MỘT LẦN duy nhất trong toàn bộ văn bản.\n"
    "3. CẤM BỊA KÝ TỰ (PLACEHOLDER): Tuyệt đối không sử dụng các ký tự đại diện hoặc giữ chỗ như [X], [Y], [Z], [Nghị định...]. Nếu tài liệu không ghi rõ số điều/khoản/năm, hãy viết cụ thể bằng chữ: 'Theo quy định của văn bản...' hoặc 'Tài liệu không nêu rõ số điều'.\n"
    "4. ĐÚNG PHÂN LOẠI: Thông tin thuộc nhóm nào thì chỉ viết vào nhóm đó (Ví dụ: Tiền hỗ trợ đào tạo/quản trị không được xếp vào nhóm ưu đãi đất đai).\n\n"
    "Bạn BẮT BUỘC phải trình bày câu trả lời nghiêm ngặt theo đúng cấu trúc 4 phần sau (Không được tự ý thêm, bớt hoặc đổi tên phần):\n"
    "1. Trả lời trực tiếp: Đưa ra câu trả lời kết luận ngắn gọn, tổng quan trong từ 1 đến 2 câu (Ví dụ: Khẳng định Có/Không, Mức phạt cụ thể là bao nhiêu, hoặc Thời gian tối đa là bao lâu).\n"
    "2. Phân tích chi tiết: Liệt kê chi tiết các điều kiện, tiêu chuẩn hoặc các bước thực hiện dưới dạng các đầu dòng súc tích. TUYỆT ĐỐI không lặp lại câu kết luận đã viết ở phần 1.\n"
    "3. Căn cứ pháp lý: Chỉ rõ Tên văn bản luật và Số hiệu điều/khoản trích xuất được từ ngữ cảnh. Nếu ngữ cảnh không có, ghi rõ 'Chưa có căn cứ điều khoản cụ thể trong tài liệu'.\n"
    "4. Hạn chế của dữ liệu: Nêu rõ những thông tin mà câu hỏi yêu cầu nhưng tài liệu được cung cấp chưa làm rõ hoặc còn thiếu (Nếu tài liệu đã đầy đủ, ghi 'Không có')."
)

USER_CONTENT_TEMPLATE = (
    "TÀI LIỆU THAM KHẢO CUNG CẤP:\n"
    "=========================================\n"
    "{context}\n"
    "=========================================\n\n"
    "{warning_msg}"
    "CÂU HỎI: {query}\n\n"
    "Yêu cầu trả lời: Hãy phân tích kỹ tài liệu tham khảo trên, suy luận logic để phân phối thông tin và trả lời chính xác theo cấu trúc 4 phần đã quy định ở trên."
)


# ==========================================
# File: retrieval/retriever.py
# ==========================================

"""
local_retriever.py
==================
Retriever chạy hoàn toàn LOCAL trên SQLite (local_chunks.db).
Không cần kết nối Supabase/internet.

Hỗ trợ:
  - FTS: SQLite FTS5 BM25 native (ultra-fast)
  - Vector: cosine similarity trên BLOB embeddings
  - Hybrid: RRF kết hợp FTS + vector

Dùng để test offline khi Supabase không accessible.
"""

import os
import sys
import re
import io
import time
import sqlite3
import numpy as np
from typing import List, Optional, Literal


LOCAL_DB_PATH = os.path.join(PROJECT_ROOT, "database", "local_chunks.db")

# ─── Reuse RetrievalResult từ retriever.py ────────────────────────────────────
from dataclasses import dataclass, field

@dataclass
class RetrievalResult:
    chunk_id: str
    document_id: int
    chunk_index: int
    content: str
    doc_number: str
    title: str
    legal_type: str
    score: float
    source: str
    article_hint: Optional[str] = None
    best_chunk_content: Optional[str] = None

    def format_relevant_doc(self) -> str:
        return f"{self.doc_number}|{self.legal_type} {self.doc_number} {self.title}"

    def format_relevant_article(self) -> str:
        doc_str = self.format_relevant_doc()
        article = self.article_hint or "Toàn bộ"
        return f"{doc_str}|{article}"

_ARTICLE_PATTERN = re.compile(
    r"(?:^|\s)(Điều\s+\d+[a-zA-Z]?)",
    re.MULTILINE | re.UNICODE
)

def extract_article_hint(content: str) -> Optional[str]:
    m = _ARTICLE_PATTERN.search(content)
    if m:
        return m.group(1).strip()
    return None



# ─── Parse metadata từ chunk content header ───────────────────────────────────
# Format: "Văn bản: Nghị định 39/2018/NĐ-CP về hướng dẫn ... (39/2018/NĐ-CP)"
_CONTENT_HEADER_RE = re.compile(
    r'V\u0103n b\u1ea3n:\s*(.+?)\s*\(([^)]+)\)',
    re.MULTILINE | re.UNICODE | re.DOTALL
)

def _parse_meta_from_content(content: str) -> dict:
    """Parse document metadata từ dòng header trong chunk content."""
    m = _CONTENT_HEADER_RE.search(content[:400])
    if not m:
        return {"document_number": "", "title": "", "legal_type": ""}

    full_title = m.group(1).strip()
    full_title = re.sub(r'\s+', ' ', full_title)
    doc_number = m.group(2).strip()

    # Tách legal_type (loại văn bản)
    legal_type_match = re.match(
        r'^(Lu\u1eadt|B\u1ed9 lu\u1eadt|Ngh\u1ecb \u0111\u1ecbnh|Th\u00f4ng t\u01b0|'
        r'Quy\u1ebft \u0111\u1ecbnh|Ngh\u1ecb quy\u1ebft|Ph\u00e1p l\u1ec7nh|'
        r'C\u00f4ng v\u0103n|Th\u00f4ng b\u00e1o|Ch\u1ec9 th\u1ecb)\b',
        full_title, re.UNICODE
    )
    legal_type = legal_type_match.group(1) if legal_type_match else ""

    # Title = xóa "<legal_type> <doc_number>" ở đầu
    title = full_title
    if legal_type and doc_number:
        title = re.sub(
            rf'^{re.escape(legal_type)}\s+{re.escape(doc_number)}\s*',
            '', title
        ).strip()

    return {
        "document_number": doc_number,
        "title": title,
        "legal_type": legal_type,
    }


def _tokenize(text: str) -> List[str]:
    """Tokenize đơn giản cho fallback search."""
    tokens = re.split(r'[\s\.,;:!\?\"\'()\[\]{}\-/\\]+', text.lower())
    return [t for t in tokens if len(t) >= 2]


class LegalRetriever:
    """
    Retriever hoàn toàn local, không cần internet.

    Args:
        db_path      : đường dẫn SQLite
        top_k        : số kết quả
        vector_weight: trọng số vector trong RRF hybrid
        fts_weight   : trọng số FTS trong RRF hybrid
        rrf_k        : hằng số RRF
    """

    def __init__(
        self,
        db_path: str = LOCAL_DB_PATH,
        top_k: int = 10,
        vector_weight: float = 0.3,
        fts_weight: float = 0.7,
        rrf_k: int = 60,
        use_postgres: bool = False,
        pg_conn_str: Optional[str] = None
    ):
        self.db_path = db_path
        self.top_k = top_k
        self.vector_weight = vector_weight
        self.fts_weight = fts_weight
        self.rrf_k = rrf_k
        self.use_postgres = False
        self._conn: Optional[sqlite3.Connection] = None
        self._model = None
        self._reranker = None
    # ── Kết nối ─────────────────────────────────────────────────────────────────

    
                   

    def _get_conn(self) -> sqlite3.Connection:
        if self._conn is None:
            if not os.path.exists(self.db_path):
                raise FileNotFoundError(f"SQLite DB not found: {self.db_path}")
            self._conn = sqlite3.connect(self.db_path, check_same_thread=False)
            self._conn.row_factory = sqlite3.Row

            # SQLite performance tuning
            try:
                cur = self._conn.cursor()
                cur.execute("PRAGMA journal_mode=WAL;")
                cur.execute("PRAGMA synchronous=OFF;")
                cur.execute("PRAGMA cache_size=-2000000;") # 2GB cache
                cur.execute("PRAGMA temp_store=MEMORY;")
                cur.execute("PRAGMA mmap_size=3000000000;") # 3GB memory map
            except Exception as e:
                print(f"[local] Warning: Failed to apply SQLite PRAGMAs: {e}")
        return self._conn

    def close(self):
        if self._conn:
            self._conn.close()
            self._conn = None

    # ── Embedding model ─────────────────────────────────────────────────────────

    def _get_model(self):
        if self._model is None:
            import os
            from sentence_transformers import SentenceTransformer
            import torch
            device = "cuda" if torch.cuda.is_available() else "cpu"
            print(f"[local] Loading bkai-bi-encoder on device '{device}'...", flush=True)
            t0 = time.time()
            self._model = SentenceTransformer(
                "bkai-foundation-models/vietnamese-bi-encoder", device=device
            )
            print(f"[local] Model loaded in {time.time()-t0:.1f}s", flush=True)
        return self._model

    def embed_query(self, query: str) -> np.ndarray:
        model = self._get_model()
        vec = model.encode([query], show_progress_bar=False)[0]
        return vec.astype(np.float32)

    # ── FTS5 check ──────────────────────────────────────────────────────────────

    def _has_fts5_index(self) -> bool:
        conn = self._get_conn()
        cur = conn.cursor()
        cur.execute(
            "SELECT name FROM sqlite_master WHERE type='table' AND name='chunks_fts5';"
        )
        result = cur.fetchone()
        if result is None:
            return False
        # Verify schema is correct (no content_rowid='id' bug)
        cur.execute("SELECT sql FROM sqlite_master WHERE name='chunks_fts5';")
        sql = cur.fetchone()
        if sql:
            fts_sql = sql[0] or ""
            # Bad schema: content_rowid='id' where id is TEXT
            if "content_rowid='id'" in fts_sql:
                return False
        return True

    # ── FTS Search ──────────────────────────────────────────────────────────────

    def fts_search(self, query: str, top_k: Optional[int] = None) -> List[RetrievalResult]:
        """
        FTS search với SQLite FTS5 BM25 native.
        Sử dụng cấu trúc AND/OR thông minh dựa trên phân tích ngữ nghĩa để tránh gây nhiễu.
        Fallback sang LIKE search nếu FTS5 index chưa được tạo đúng.
        """

        k = top_k or self.top_k

        conn = self._get_conn()
        cur = conn.cursor()

        # Helper function to execute two-step FTS search to avoid SQLite slow JOIN behavior
        def _execute_fts_query(match_expr: str, limit: int) -> List[RetrievalResult]:
            try:
                cur.execute("""
                    SELECT rowid, -bm25(chunks_fts5) AS score
                    FROM chunks_fts5
                    WHERE chunks_fts5 MATCH ?
                    ORDER BY bm25(chunks_fts5)
                    LIMIT ?;
                """, (match_expr, limit))
                rows = cur.fetchall()
                if not rows:
                    return []

                rowids = [r[0] for r in rows]
                scores = {r[0]: float(r[1]) if r[1] is not None else 0.0 for r in rows}

                placeholders = ",".join("?" for _ in rowids)
                cur.execute(f"""
                    SELECT rowid, id, document_id, chunk_index, content
                    FROM document_chunks
                    WHERE rowid IN ({placeholders})
                """, rowids)
                details = cur.fetchall()

                results_list = []
                for row in details:
                    r_id, cid, doc_id, chunk_idx, content = row
                    score = scores.get(r_id, 0.0)
                    meta = _parse_meta_from_content(content or "")
                    results_list.append(RetrievalResult(
                        chunk_id=str(cid),
                        document_id=doc_id,
                        chunk_index=chunk_idx,
                        content=content or "",
                        doc_number=meta["document_number"],
                        title=meta["title"],
                        legal_type=meta["legal_type"],
                        score=score,
                        source="fts",
                        article_hint=extract_article_hint(content or ""),
                    ))
                results_list.sort(key=lambda x: x.score, reverse=True)
                return results_list
            except Exception as e:
                print(f"[local][FTS5] MATCH query error: {e}")
                return []

        if self._has_fts5_index():
            # Xây dựng câu truy vấn FTS cấu trúc AND/OR
            q_lower = query.lower()

            # Phân nhóm thực thể (Subjects) và Chủ đề/Hành động (Topics/Actions)
            subjects = {
                "incubator_coworking": {
                    "keywords": ["ươm tạo", "làm việc chung"],
                    "syns": ['"ươm tạo"', '"cơ sở ươm tạo"', '"khu làm việc chung"', '"làm việc chung"']
                },
                "sme": {
                    "keywords": ["doanh nghiệp nhỏ và vừa", "nhỏ và vừa", "sme", "siêu nhỏ"],
                    "syns": ['"doanh nghiệp nhỏ và vừa"', '"nhỏ và vừa"', '"doanh nghiệp nhỏ"', '"doanh nghiệp siêu nhỏ"']
                },
                "labor": {
                    "keywords": ["nhân viên", "lao động", "người lao động", "thử việc", "ký hợp đồng", "sa thải", "đuổi việc", "nghỉ việc", "lương"],
                    "syns": ['"người lao động"', '"lao động"', '"nhân viên"', '"hợp đồng lao động"']
                }
            }

            topics = {
                "tax_land": {
                    "keywords": ["thuế", "đất đai", "mặt bằng", "tiền thuê"],
                    "syns": ['"thuế"', '"đất đai"', '"mặt bằng"', '"thuê đất"', '"ưu đãi thuế"', '"tiền sử dụng đất"']
                },
                "bidding": {
                    "keywords": ["đấu thầu", "nhà thầu", "gói thầu"],
                    "syns": ['"đấu thầu"', '"nhà thầu"', '"lựa chọn nhà thầu"', '"gói thầu"', '"xếp hạng hồ sơ"']
                },
                "degrees_holding": {
                    "keywords": ["giữ", "bằng", "văn bằng", "chứng chỉ", "giấy tờ", "bằng cấp"],
                    "syns": ['"giữ"', '"thu giữ"', '"tạm giữ"', '"văn bằng"', '"chứng chỉ"', '"giấy tờ tùy thân"', '"bản chính"', '"bằng cấp"']
                }
            }

            active_subjects = []
            for s_name, s_info in subjects.items():
                if any(kw in q_lower for kw in s_info["keywords"]):
                    active_subjects.append(s_info["syns"])

            active_topics = []
            for t_name, t_info in topics.items():
                if any(kw in q_lower for kw in t_info["keywords"]):
                    active_topics.append(t_info["syns"])

            clauses = []
            if active_subjects:
                flat_subjects = []
                for s in active_subjects:
                    flat_subjects.extend(s)
                flat_subjects = list(dict.fromkeys(flat_subjects))
                clauses.append("(" + " OR ".join(flat_subjects) + ")")

            if active_topics:
                flat_topics = []
                for t in active_topics:
                    flat_topics.extend(t)
                flat_topics = list(dict.fromkeys(flat_topics))
                clauses.append("(" + " OR ".join(flat_topics) + ")")

            fts_expr = ""
            if clauses:
                fts_expr = " AND ".join(clauses)
                print(f"[local][FTS5] Structured query: {fts_expr}")
                results = _execute_fts_query(fts_expr, k)
                if results:
                    return results
                print(f"[local][FTS5] Structured query returned 0 results. Trying fallback OR...")

            # Fallback sang OR đơn giản của các từ khóa có nghĩa (loại bỏ stopword)
            GRAMMAR_STOPWORDS = {
                "và", "hoặc", "nhưng", "vì", "nên", "của", "các", "những", "là", "có",
                "trong", "tại", "để", "theo", "được", "bị", "cho", "ra", "vào", "lên",
                "xuống", "do", "từ", "đến", "bằng", "with", "với", "về", "như", "thì",
                "mà", "khi", "gì", "này", "đó", "kia", "nọ", "thế", "vậy", "nào", "ai",
                "đâu", "cái", "con", "chiếc", "nếu", "sẽ", "phải", "sao", "thế"
            }
            words = [w.lower().strip() for w in query.split()]
            terms = []
            for w in words:
                w = re.sub(r'[^\w\s\u00C0-\u024F\u1E00-\u1EFF]', '', w).strip()
                if w and len(w) >= 3 and w not in GRAMMAR_STOPWORDS:
                    terms.append(f'"{w}"')

            if terms:
                fallback_expr = " OR ".join(terms)
                print(f"[local][FTS5] Fallback OR query: {fallback_expr}")
                results = _execute_fts_query(fallback_expr, k)
                if results:
                    return results

            print("[local][FTS5] 0 results from FTS5. Trying LIKE fallback...")

        # LIKE fallback search (slower, for queries without diacritics)
        print("[local][FTS] Using LIKE fallback search...")
        keywords = [w for w in query.split() if len(w) >= 3][:5]
        if not keywords:
            return []

        like_pattern = f"%{keywords[0]}%"
        cur.execute("""
            SELECT id, document_id, chunk_index, content
            FROM document_chunks
            WHERE content LIKE ?
            LIMIT ?;
        """, (like_pattern, k * 3))
        rows = cur.fetchall()

        results = []
        for row in rows:
            cid, doc_id, chunk_idx, content = row[0], row[1], row[2], row[3] or ""
            score = float(sum(1 for kw in keywords if kw in content))
            if score == 0:
                continue
            meta = _parse_meta_from_content(content)
            results.append(RetrievalResult(
                chunk_id=str(cid),
                document_id=doc_id,
                chunk_index=chunk_idx,
                content=content,
                doc_number=meta["document_number"],
                title=meta["title"],
                legal_type=meta["legal_type"],
                score=score,
                source="fts",
                article_hint=extract_article_hint(content),
            ))
        results.sort(key=lambda x: -x.score)
        return results[:k]

    # ── FAISS Vector Search ───────────────────────────────────────────────────────

    def _get_faiss_index(self):
        """Lấy hoặc tự động build FAISS index từ SQLite database."""
        import faiss
        index_path = os.path.join(os.path.dirname(self.db_path), "local_chunks.index")

        if not hasattr(self, "_faiss_index") or self._faiss_index is None:
            if os.path.exists(index_path):
                print(f"[local] Loading FAISS index from {index_path}...", flush=True)
                t0 = time.time()
                self._faiss_index = faiss.read_index(index_path)
                print(f"[local] FAISS index loaded in {time.time()-t0:.2f}s", flush=True)
            else:
                print(f"[local] FAISS index not found. Building FAISS index from {self.db_path} (one-time setup)...", flush=True)
                t0 = time.time()
                conn = self._get_conn()
                cur = conn.cursor()
                cur.execute("SELECT rowid, embedding FROM document_chunks WHERE embedding IS NOT NULL;")
                rowids = []
                embeddings = []
                for rowid, emb_bytes in cur:
                    if not emb_bytes:
                        continue
                    emb = np.frombuffer(emb_bytes, dtype=np.float32)
                    rowids.append(rowid)
                    embeddings.append(emb)

                if not embeddings:
                    raise ValueError("No embeddings found in database to build FAISS index!")

                embeddings = np.array(embeddings, dtype=np.float32)
                rowids = np.array(rowids, dtype=np.int64)

                # Chuẩn hóa vector L2 để tìm kiếm Cosine Similarity bằng Inner Product
                norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
                norms[norms == 0] = 1.0
                embeddings = embeddings / norms

                dim = embeddings.shape[1]
                quantizer = faiss.IndexFlatIP(dim)
                index = faiss.IndexIDMap(quantizer)
                index.add_with_ids(embeddings, rowids)

                # Lưu index để các lần sau dùng ngay
                faiss.write_index(index, index_path)
                self._faiss_index = index
                print(f"[local] FAISS index built and saved in {time.time()-t0:.2f}s", flush=True)

        return self._faiss_index

    def vector_search(self, query: str, top_k: Optional[int] = None) -> List[RetrievalResult]:
        """Tìm kiếm tương đồng vector bằng FAISS index (tải và tìm cực nhanh)."""
        k = top_k or self.top_k

        query_vec = self.embed_query(query)
        query_norm = np.linalg.norm(query_vec)
        if query_norm == 0:
            return []

        # Chuẩn hóa query vector và định dạng cho FAISS
        query_vec = query_vec / query_norm
        query_vec = np.ascontiguousarray(query_vec.reshape(1, -1), dtype=np.float32)

        index = self._get_faiss_index()
        scores, indices = index.search(query_vec, k)
        scores = scores[0]
        indices = indices[0]

        valid_indices = [int(idx) for idx in indices if idx != -1]
        if not valid_indices:
            return []

        # Truy vấn SQLite để lấy thông tin các chunks tương ứng với rowid
        conn = self._get_conn()
        cur = conn.cursor()
        placeholders = ",".join("?" for _ in valid_indices)
        cur.execute(f"""
            SELECT rowid, id, document_id, chunk_index, content
            FROM document_chunks
            WHERE rowid IN ({placeholders});
        """, valid_indices)

        rows = cur.fetchall()
        row_map = {row["rowid"]: row for row in rows}

        results = []
        for idx, score in zip(indices, scores):
            idx = int(idx)
            if idx not in row_map:
                continue
            row = row_map[idx]
            cid = row["id"]
            doc_id = row["document_id"]
            chunk_idx = row["chunk_index"]
            content = row["content"] or ""

            meta = _parse_meta_from_content(content)
            results.append(RetrievalResult(
                chunk_id=str(cid),
                document_id=doc_id,
                chunk_index=chunk_idx,
                content=content,
                doc_number=meta["document_number"],
                title=meta["title"],
                legal_type=meta["legal_type"],
                score=float(score),
                source="vector",
                article_hint=extract_article_hint(content),
            ))
        return results

    # ── Hybrid (RRF) ────────────────────────────────────────────────────────────

    def hybrid_search(self, query: str, top_k: Optional[int] = None) -> List[RetrievalResult]:
        """Kết hợp FTS + vector bằng Reciprocal Rank Fusion."""
        k = top_k or self.top_k
        fetch_k = k * 3

        fts_results    = self.fts_search(query,    top_k=fetch_k)
        vector_results = self.vector_search(query, top_k=fetch_k)

        chunk_map: dict = {}
        rrf_scores: dict = {}

        def add_ranked(results: List[RetrievalResult], weight: float):
            for rank, r in enumerate(results, start=1):
                cid = r.chunk_id
                if cid not in chunk_map:
                    chunk_map[cid] = r
                    rrf_scores[cid] = 0.0
                rrf_scores[cid] += weight / (rank + self.rrf_k)

        add_ranked(fts_results,    self.fts_weight)
        add_ranked(vector_results, self.vector_weight)

        sorted_ids = sorted(rrf_scores, key=rrf_scores.get, reverse=True)
        results = []
        for cid in sorted_ids[:k]:
            r = chunk_map[cid]
            results.append(RetrievalResult(
                chunk_id=r.chunk_id,
                document_id=r.document_id,
                chunk_index=r.chunk_index,
                content=r.content,
                doc_number=r.doc_number,
                title=r.title,
                legal_type=r.legal_type,
                score=rrf_scores[cid],
                source="hybrid",
                article_hint=r.article_hint,
            ))
        return results

    def rerank(self, query: str, results: List[RetrievalResult], top_k: Optional[int] = None) -> List[RetrievalResult]:
        if not results:
            return []
    
        if self._reranker is None:
            import torch
            import time
            from sentence_transformers import CrossEncoder
    
            device = "cuda" if torch.cuda.is_available() else "cpu"
    
            print(
                f"[local] Loading reranker model BAAI/bge-reranker-v2-m3 on device '{device}'...",
                flush=True
            )
    
            t0 = time.time()
    
            automodel_args = {}
            if device == "cuda":
                automodel_args = {
                    "torch_dtype": torch.float16,
                    "low_cpu_mem_usage": True
                }
    
            self._reranker = CrossEncoder(
                "BAAI/bge-reranker-v2-m3",
                device=device,
                automodel_args=automodel_args
            )
    
            print(
                f"[local] Reranker model loaded in {time.time() - t0:.1f}s",
                flush=True
            )

        pairs = []
        for r in results:
            best_text = getattr(r, 'best_chunk_content', None) or r.content

            # Thêm metadata boosting để tăng độ chính xác (rất hiệu quả với văn bản pháp luật)
            article_str = f"Điều: {r.article_hint}\n" if r.article_hint else ""
            doc_text = f"Văn bản: {r.legal_type} {r.doc_number} - {r.title}\n{article_str}Nội dung:\n{best_text}"

            pairs.append([query, doc_text])

        t_rerank = time.time()
        scores = self._reranker.predict(pairs, show_progress_bar=False)
        print(f"[local] Reranking took {time.time() - t_rerank:.2f}s for {len(pairs)} pairs.")

        LEGAL_WEIGHT = {
            "Luật": 1.0,
            "Bộ luật": 1.0,
            "Nghị quyết": 0.95,
            "Nghị định": 0.85,
            "Thông tư": 0.75,
        }

        for r, score in zip(results, scores):
            rerank_score = float(score)
            # Normalize PhoRanker output to positive if it outputs logits
            if rerank_score < 0 or rerank_score > 1.0:
                import math
                rerank_score = 1 / (1 + math.exp(-max(min(rerank_score, 100), -100)))

            weight = LEGAL_WEIGHT.get(r.legal_type, 0.7)
            final_score = rerank_score * weight

            r.score = final_score
            r.source = f"rerank({r.source})"

        results.sort(key=lambda x: x.score, reverse=True)

        # Lọc MMR Diversity: max 2 chunk mỗi document
        filtered_results = []
        doc_count = {}
        for r in results:
            doc_id = r.document_id
            if doc_count.get(doc_id, 0) < 2:
                filtered_results.append(r)
                doc_count[doc_id] = doc_count.get(doc_id, 0) + 1

        if top_k:
            return filtered_results[:top_k]
        return filtered_results

    def are_chunks_duplicate(self, c1: RetrievalResult, c2: RetrievalResult) -> bool:
        """Kiểm tra xem hai chunk có trùng lặp nội dung đáng kể hay không."""
        if c1.document_id == c2.document_id and c1.chunk_index == c2.chunk_index:
            return True

        # Trích xuất phần nội dung thực tế để so sánh
        body1 = c1.content.split("| Nội dung:")[-1].strip()
        body2 = c2.content.split("| Nội dung:")[-1].strip()

        def normalize(text):
            text = text.lower()
            text = re.sub(r'[^\w\s]', '', text)
            return set(text.split())

        words1 = normalize(body1)
        words2 = normalize(body2)
        if not words1 or not words2:
            return False

        intersection = words1.intersection(words2)
        union = words1.union(words2)
        jaccard = len(intersection) / len(union)

        # Jaccard score cao => trùng lặp
        if jaccard > 0.8:
            return True

        # Nếu trùng Điều/Khoản luật và độ tương đồng tương đối cao => trùng lặp (ví dụ giữa Luật và VBHN)
        if c1.article_hint and c2.article_hint and c1.article_hint == c2.article_hint:
            if "điều" in c1.article_hint.lower() or "khoản" in c1.article_hint.lower():
                if jaccard > 0.6:
                    return True
        return False

    # ── Entry point ─────────────────────────────────────────────────────────────


    def expand_query(self, query: str) -> str:
        """Query Expansion bằng rule-based dictionary."""
        q_lower = query.lower()
        expanded_terms = []

        intents = {
            "hỗ trợ": ["ưu đãi", "chính sách hỗ trợ", "khuyến khích"],
            "ưu đãi": ["miễn", "giảm", "hỗ trợ"],
            "miễn": ["giảm", "không phải nộp", "ưu đãi"],
            "giảm": ["miễn", "ưu đãi thuế"],
            "đất đai": ["tiền thuê đất", "tiền sử dụng đất", "mặt bằng", "đất"],
            "thuế": ["thuế thu nhập doanh nghiệp", "thuế tndn", "miễn thuế", "giảm thuế"]
        }

        for k, v in intents.items():
            if k in q_lower:
                expanded_terms.extend(v)

        if expanded_terms:
            new_terms = [t for t in expanded_terms if t not in q_lower]
            if new_terms:
                return query + " " + " ".join(new_terms)
        return query

    def expand_to_parent_article(self, results: list) -> list:
        """Parent Retrieval: Lấy toàn bộ nội dung của Điều luật từ các chunk ban đầu."""
        if not results:
            return []

        target_articles = {}
        hit_indexes = {}
        for r in results:
            if not r.article_hint:
                continue
            doc_id = r.document_id
            if doc_id not in target_articles:
                target_articles[doc_id] = set()
            target_articles[doc_id].add(r.article_hint)

            # Ghi nhớ vị trí chunk_index đã hit
            key = (doc_id, r.article_hint)
            if key not in hit_indexes:
                hit_indexes[key] = set()
            hit_indexes[key].add(r.chunk_index)

        if not target_articles:
            return results

        doc_ids = list(target_articles.keys())
        conn = self._get_conn()
        cur = conn.cursor()
        placeholders = ",".join("?" for _ in doc_ids)
        cur.execute(f"""
            SELECT document_id, chunk_index, content
            FROM document_chunks
            WHERE document_id IN ({placeholders})
            ORDER BY document_id, chunk_index
        """, doc_ids)
        all_chunks = cur.fetchall()
        cur.close()

        doc_chunks = {}
        for row in all_chunks:
            did = row["document_id"]
            if did not in doc_chunks:
                doc_chunks[did] = []
            doc_chunks[did].append(row)

        expanded_articles = {}

        for did, rows in doc_chunks.items():
            current_article = None
            for row in rows:
                content = row["content"] or ""
                # Rely on global extract_article_hint
                hint = extract_article_hint(content)
                if hint:
                    current_article = hint

                if current_article and current_article in target_articles[did]:
                    key = (did, current_article)

                    # Windowing limit: Chỉ gộp nếu chunk nằm gần vị trí hit (+/- 2 chunks)
                    # Chống phình to context đối với các Điều omnibus (VD: "sửa đổi, bổ sung một số điều")
                    hits = hit_indexes.get(key, set())
                    is_near_hit = False
                    for h in hits:
                        if abs(h - row["chunk_index"]) <= 2:
                            is_near_hit = True
                            break

                    if is_near_hit:
                        if key not in expanded_articles:
                            expanded_articles[key] = []
                        expanded_articles[key].append(content)

        final_results = []
        seen_keys = set()

        for r in results:
            if not r.article_hint:
                final_results.append(r)
                continue

            key = (r.document_id, r.article_hint)
            if key in seen_keys:
                continue
            seen_keys.add(key)

            if key in expanded_articles:
                # Lưu lại nội dung của best chunk (chunk ban đầu có điểm cao nhất) để dùng cho Rerank
                r.best_chunk_content = r.content

                full_content = "\n...\n".join(expanded_articles[key])
                r.content = full_content
                r.source += "_parent_expanded"
                final_results.append(r)
            else:
                final_results.append(r)

        return final_results

    def retrieve(
        self,
        query: str,
        mode: Literal["vector", "fts", "hybrid"] = "fts",
        top_k: Optional[int] = None,
        rerank: bool = True,
    ) -> List[RetrievalResult]:
        t0 = time.time()
        k = top_k or self.top_k

        # 1. Query Expansion (Intent Understanding)
        expanded_query = self.expand_query(query)

        # 2. Truy xuất danh sách ứng viên (candidate pool)
        if rerank:
            fetch_k = 50
        else:
            fetch_k = max(50, k * 4)

        if mode == "vector":
            results = self.vector_search(expanded_query, top_k=fetch_k)
        elif mode == "fts":
            results = self.fts_search(expanded_query, top_k=fetch_k)
        else:
            results = self.hybrid_search(expanded_query, top_k=fetch_k)

        # 3. Tiến hành lọc trùng lặp trước để loại bỏ nhiễu
        unique_results = []
        for r in results:
            is_dup = False
            for ur in unique_results:
                if self.are_chunks_duplicate(r, ur):
                     is_dup = True
                     break
            if not is_dup:
                unique_results.append(r)

        # 4. Parent Retrieval (Thay thế cho aggregate_chunks_into_articles)
        article_results = self.expand_to_parent_article(unique_results)

        # 5. Rerank và lọc
        if rerank:
            results = self.rerank(query, article_results, top_k=None)
        else:
            article_results.sort(key=lambda x: x.score, reverse=True)
            results = article_results

        # 4. Áp dụng Absolute & Relative Score Thresholds (Dynamic Thresholding)
        if results:
            RERANK_THRESHOLD = 0.40
            best_score = results[0].score

            relative_threshold_ratio = 0.65

            # Dynamic gap filtering: Nếu top 1 vượt trội hoàn toàn top 2
            if len(results) > 1:
                score_1 = results[0].score
                score_2 = results[1].score
                if (score_1 - score_2) > 0.15:
                    relative_threshold_ratio = 0.8
                elif score_1 > 0.9:
                    relative_threshold_ratio = 0.75

            filtered_by_score = []
            for r in results:
                # Giữ chunk nếu thỏa mãn ngưỡng tuyệt đối và ngưỡng tương đối động
                if r.score >= RERANK_THRESHOLD and r.score >= best_score * relative_threshold_ratio:
                    filtered_by_score.append(r)
            results = filtered_by_score

        # 5. Article-level Dedup (Giảm context thừa)
        seen_articles = set()
        deduped = []
        for r in results:
            article_id = r.format_relevant_article()
            if article_id in seen_articles:
                continue
            seen_articles.add(article_id)
            deduped.append(r)
        results = deduped

        # 6. Intent-aware (Query-aware) Filter
        try:
            import underthesea
            tags = underthesea.pos_tag(query)
            important_terms = [word.lower() for word, tag in tags if tag in ['N', 'Np', 'V', 'A', 'M'] and len(word) > 1]
            if not important_terms:
                important_terms = [w.lower() for w in query.split() if len(w) > 2]
        except Exception:
            important_terms = [w.lower() for w in query.split() if len(w) > 2]

        filtered_final = []
        for r in results:
            content_lower = r.content.lower()
            # Chunk phải chứa ít nhất một term quan trọng (intent keywords) từ câu hỏi
            if any(term in content_lower for term in important_terms):
                filtered_final.append(r)

        # Fallback nếu filter quá ngặt nghèo drop hết tất cả kết quả
        if not filtered_final and results:
            filtered_final = results

        results = filtered_final[:k]

        elapsed = time.time() - t0
        print(f"[local] mode={mode} | rerank={rerank} | {len(results)} results | {elapsed:.2f}s")
        return results

    @staticmethod
    def aggregate_by_document(results: List[RetrievalResult]) -> dict:
        """Nhóm kết quả theo document_id."""
        docs: dict = {}
        for r in results:
            if r.document_id not in docs:
                docs[r.document_id] = {
                    "doc_number": r.doc_number,
                    "title": r.title,
                    "legal_type": r.legal_type,
                    "chunks": [],
                    "max_score": 0.0,
                    "articles": set(),
                }
            docs[r.document_id]["chunks"].append(r)
            docs[r.document_id]["max_score"] = max(
                docs[r.document_id]["max_score"], r.score
            )
            if r.article_hint:
                docs[r.document_id]["articles"].add(r.article_hint)
        for doc in docs.values():
            try:
                doc["articles"] = sorted(
                    doc["articles"],
                    key=lambda x: int(re.search(r'\d+', x).group())
                )
            except Exception:
                doc["articles"] = sorted(doc["articles"])
        return docs


# ─── Quick CLI test ────────────────────────────────────────────────────────────

if __name__ == "__main__":
    if sys.platform == "win32":
        sys.stdout = io.TextIOWrapper(sys.stdout.buffer, encoding="utf-8", errors="replace")

    query = sys.argv[1] if len(sys.argv) > 1 else "doanh nghiệp nhỏ và vừa"
    mode  = sys.argv[2] if len(sys.argv) > 2 else "fts"
    top_k = int(sys.argv[3]) if len(sys.argv) > 3 else 5

    print(f"Query : {query}")
    print(f"Mode  : {mode}")
    print(f"Top-K : {top_k}")
    print("-" * 60)

    r = LegalRetriever(top_k=top_k)
    results = r.retrieve(query, mode=mode, top_k=top_k)
    r.close()

    for i, res in enumerate(results, 1):
        print(f"\n[{i}] {res.doc_number} | {res.legal_type}")
        print(f"    Tieu de : {res.title[:70]}")
        print(f"    Chunk#{res.chunk_index} | {res.article_hint or '-'} | score={res.score:.4f}")
        print(f"    Content : {res.content[:200]}...")

# ==========================================
# File: llm/generator.py
# ==========================================

import os
import time
from typing import Optional

class PipelineGenerator:
    """
    Step 8: Qwen3-8B
    Sinh câu trả lời dựa trên query đã xử lý và context đã được nén.
    """
    def __init__(self, model_name: str = "Qwen/Qwen3-8B-Instruct"):
        self.model_name = model_name
        self._tokenizer = None
        self._model = None

    def _lazy_load(self):
        if self._model is not None:
            return

        import torch
        from transformers import AutoTokenizer, AutoModelForCausalLM

        is_offline = os.environ.get("HF_HUB_OFFLINE") == "1"

        print(f"[generator] Loading model '{self.model_name}' (Offline={is_offline})...", flush=True)
        t0 = time.time()

        try:
            self._tokenizer = AutoTokenizer.from_pretrained(
                self.model_name,
                trust_remote_code=True,
                local_files_only=is_offline
            )

            device = "cuda" if torch.cuda.is_available() else "cpu"
            dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16

            print(f"[generator] Device={device}, Dtype={dtype}...", flush=True)

            self._model = AutoModelForCausalLM.from_pretrained(
                self.model_name,
                torch_dtype=dtype,
                device_map="auto",
                trust_remote_code=True,
                local_files_only=is_offline
            )
            print(f"[generator] Model loaded in {time.time() - t0:.1f}s", flush=True)
        except Exception as e:
            print(f"[generator] Failed to load model: {e}")
            raise

    def generate_direct(self, prompt: str, max_new_tokens: int = 1024) -> str:
        """Sinh kết quả trực tiếp từ prompt (dùng cho Rewrite)."""
        self._lazy_load()
        messages = [{"role": "user", "content": prompt}]
        chat_prompt = self._tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = self._tokenizer([chat_prompt], return_tensors="pt").to(self._model.device)
        outputs = self._model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
        generated_ids = [output_ids[len(input_ids):] for input_ids, output_ids in zip(inputs.input_ids, outputs)]
        return self._tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0].strip()

    def generate_answer(self, query: str, context: str, warning_msg: Optional[str] = None, max_new_tokens: int = 1024) -> str:
        """Sinh câu trả lời cuối cùng dựa trên context."""
        if not context.strip():
            return "Không tìm thấy tài liệu luật pháp liên quan để làm căn cứ trả lời."

        self._lazy_load()


        system_prompt = SYSTEM_PROMPT

        warning_str = f"LƯU Ý QUAN TRỌNG TỪ HỆ THỐNG: {warning_msg}\n\n" if warning_msg else ""
        user_content = USER_CONTENT_TEMPLATE.format(
            context=context,
            warning_msg=warning_str,
            query=query
        )

        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_content}
        ]

        prompt = self._tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        inputs = self._tokenizer([prompt], return_tensors="pt").to(self._model.device)

        outputs = self._model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.1,
            repetition_penalty=1.15,
            pad_token_id=self._tokenizer.eos_token_id
        )

        generated_ids = [
            output_ids[len(input_ids):] for input_ids, output_ids in zip(inputs.input_ids, outputs)
        ]

        response = self._tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
        return response.strip()

# ==========================================
# File: retrieval/query_rewrite.py
# ==========================================

import re

class QueryRewriter:
    """
    Step 2: Query Rewrite.
    Mở rộng câu hỏi bằng các từ đồng nghĩa hoặc viết lại câu hỏi rõ ràng hơn.
    """
    def __init__(self, use_llm=False, llm_generator=None):
        self.use_llm = use_llm
        self.llm_generator = llm_generator

        # Simple rule-based synonym dictionary
        self.synonyms = {
            "sme": ["doanh nghiệp nhỏ và vừa", "nhỏ và vừa", "doanh nghiệp siêu nhỏ"],
            "sa thải": ["đuổi việc", "chấm dứt hợp đồng lao động", "kỷ luật"],
            "thai sản": ["nghỉ đẻ", "mang thai", "sinh con"],
            "bảo hiểm xã hội": ["bhxh", "bảo hiểm", "đóng bảo hiểm"],
            "giấy tờ": ["bản chính", "bằng cấp", "chứng chỉ", "văn bằng", "giữ giấy tờ"],
            "công đoàn": ["tổ chức công đoàn", "đại diện người lao động"]
        }

    def rewrite(self, query: str) -> str:
        """Thực hiện rewrite query."""
        if self.use_llm and self.llm_generator:
            return self._llm_rewrite(query)
        return self._rule_based_rewrite(query)

    def _rule_based_rewrite(self, query: str) -> str:
        q_lower = query.lower()
        expanded_terms = set()

        for key, syns in self.synonyms.items():
            if key in q_lower or any(s in q_lower for s in syns):
                expanded_terms.update(syns)

        if expanded_terms:
            # Combine original query with expanded terms
            expansion = " ".join(expanded_terms)
            # Avoid too long queries
            return f"{query} {expansion}"
        return query

    def _llm_rewrite(self, query: str) -> str:
        # Giả định dùng LLM để trích xuất keyword
        # Để tiết kiệm thời gian và tài nguyên, ta có thể cài đặt prompt ngắn
        prompt = (
            "Bạn là một chuyên gia luật Việt Nam.\n"
            "Hãy viết lại câu hỏi sau thành một câu truy vấn tìm kiếm ngắn gọn, "
            "chứa các từ khóa pháp lý quan trọng nhất, không dài quá 20 từ.\n\n"
            f"Câu hỏi: {query}\n\n"
            "Truy vấn tìm kiếm:"
        )
        if hasattr(self.llm_generator, 'generate_direct'):
            rewritten = self.llm_generator.generate_direct(prompt, max_new_tokens=50)
            if rewritten and len(rewritten) > 5:
                return rewritten
        return query

# ==========================================
# File: retrieval/local_retriever.py
# ==========================================

"""
local_retriever.py
==================
Retriever chạy hoàn toàn LOCAL trên SQLite (local_chunks.db).
Không cần kết nối Supabase/internet.

Hỗ trợ:
  - FTS: SQLite FTS5 BM25 native (ultra-fast)
  - Vector: cosine similarity trên BLOB embeddings
  - Hybrid: RRF kết hợp FTS + vector

Dùng để test offline khi Supabase không accessible.
"""

import os
import sys
import re
import io
import time
import sqlite3
import numpy as np
from typing import List, Optional, Literal


LOCAL_DB_PATH = os.path.join(PROJECT_ROOT, "database", "local_chunks.db")

# ─── Reuse RetrievalResult từ retriever.py ────────────────────────────────────
from dataclasses import dataclass, field

@dataclass
class RetrievalResult:
    chunk_id: str
    document_id: int
    chunk_index: int
    content: str
    doc_number: str
    title: str
    legal_type: str
    score: float
    source: str
    article_hint: Optional[str] = None
    best_chunk_content: Optional[str] = None

    def format_relevant_doc(self) -> str:
        return f"{self.doc_number}|{self.legal_type} {self.doc_number} {self.title}"

    def format_relevant_article(self) -> str:
        doc_str = self.format_relevant_doc()
        article = self.article_hint or "Toàn bộ"
        return f"{doc_str}|{article}"

_ARTICLE_PATTERN = re.compile(
    r"(?:^|\s)(Điều\s+\d+[a-zA-Z]?)",
    re.MULTILINE | re.UNICODE
)

def extract_article_hint(content: str) -> Optional[str]:
    m = _ARTICLE_PATTERN.search(content)
    if m:
        return m.group(1).strip()
    return None



# ─── Parse metadata từ chunk content header ───────────────────────────────────
# Format: "Văn bản: Nghị định 39/2018/NĐ-CP về hướng dẫn ... (39/2018/NĐ-CP)"
_CONTENT_HEADER_RE = re.compile(
    r'V\u0103n b\u1ea3n:\s*(.+?)\s*\(([^)]+)\)',
    re.MULTILINE | re.UNICODE | re.DOTALL
)

def _parse_meta_from_content(content: str) -> dict:
    """Parse document metadata từ dòng header trong chunk content."""
    m = _CONTENT_HEADER_RE.search(content[:400])
    if not m:
        return {"document_number": "", "title": "", "legal_type": ""}

    full_title = m.group(1).strip()
    full_title = re.sub(r'\s+', ' ', full_title)
    doc_number = m.group(2).strip()

    # Tách legal_type (loại văn bản)
    legal_type_match = re.match(
        r'^(Lu\u1eadt|B\u1ed9 lu\u1eadt|Ngh\u1ecb \u0111\u1ecbnh|Th\u00f4ng t\u01b0|'
        r'Quy\u1ebft \u0111\u1ecbnh|Ngh\u1ecb quy\u1ebft|Ph\u00e1p l\u1ec7nh|'
        r'C\u00f4ng v\u0103n|Th\u00f4ng b\u00e1o|Ch\u1ec9 th\u1ecb)\b',
        full_title, re.UNICODE
    )
    legal_type = legal_type_match.group(1) if legal_type_match else ""

    # Title = xóa "<legal_type> <doc_number>" ở đầu
    title = full_title
    if legal_type and doc_number:
        title = re.sub(
            rf'^{re.escape(legal_type)}\s+{re.escape(doc_number)}\s*',
            '', title
        ).strip()

    return {
        "document_number": doc_number,
        "title": title,
        "legal_type": legal_type,
    }


def _tokenize(text: str) -> List[str]:
    """Tokenize đơn giản cho fallback search."""
    tokens = re.split(r'[\s\.,;:!\?\"\'()\[\]{}\-/\\]+', text.lower())
    return [t for t in tokens if len(t) >= 2]


class LocalRetriever:
    """
    Retriever hoàn toàn local, không cần internet.

    Args:
        db_path      : đường dẫn SQLite
        top_k        : số kết quả
        vector_weight: trọng số vector trong RRF hybrid
        fts_weight   : trọng số FTS trong RRF hybrid
        rrf_k        : hằng số RRF
    """

    def __init__(
        self,
        db_path: str = LOCAL_DB_PATH,
        top_k: int = 10,
        vector_weight: float = 0.3,
        fts_weight: float = 0.7,
        rrf_k: int = 60,
    ):
        self.db_path = db_path
        self.top_k = top_k
        self.vector_weight = vector_weight
        self.fts_weight = fts_weight
        self.rrf_k = rrf_k
        self._conn: Optional[sqlite3.Connection] = None
        self._model = None
        self._reranker = None

    # ── Kết nối ─────────────────────────────────────────────────────────────────

    def _get_conn(self) -> sqlite3.Connection:
        if self._conn is None:
            if not os.path.exists(self.db_path):
                raise FileNotFoundError(f"SQLite DB not found: {self.db_path}")
            self._conn = sqlite3.connect(self.db_path, check_same_thread=False)
            self._conn.row_factory = sqlite3.Row

            # SQLite performance tuning
            try:
                cur = self._conn.cursor()
                cur.execute("PRAGMA journal_mode=WAL;")
                cur.execute("PRAGMA synchronous=OFF;")
                cur.execute("PRAGMA cache_size=-100000;") # 100MB cache
                cur.execute("PRAGMA temp_store=MEMORY;")
                cur.execute("PRAGMA mmap_size=500000000;") # 500MB memory map
            except Exception as e:
                print(f"[local] Warning: Failed to apply SQLite PRAGMAs: {e}")
        return self._conn

    def close(self):
        if self._conn:
            self._conn.close()
            self._conn = None

    # ── Embedding model ─────────────────────────────────────────────────────────

    def _get_model(self):
        if self._model is None:
            import os
            from sentence_transformers import SentenceTransformer
            import torch
            device = "cuda" if torch.cuda.is_available() else "cpu"
            print(f"[local] Loading bkai-bi-encoder on device '{device}'...", flush=True)
            t0 = time.time()
            self._model = SentenceTransformer(
                "bkai-foundation-models/vietnamese-bi-encoder", device=device
            )
            print(f"[local] Model loaded in {time.time()-t0:.1f}s", flush=True)
        return self._model

    def embed_query(self, query: str) -> np.ndarray:
        model = self._get_model()
        vec = model.encode([query], show_progress_bar=False)[0]
        return vec.astype(np.float32)

    # ── FTS5 check ──────────────────────────────────────────────────────────────

    def _has_fts5_index(self) -> bool:
        conn = self._get_conn()
        cur = conn.cursor()
        cur.execute(
            "SELECT name FROM sqlite_master WHERE type='table' AND name='chunks_fts5';"
        )
        result = cur.fetchone()
        if result is None:
            return False
        # Verify schema is correct (no content_rowid='id' bug)
        cur.execute("SELECT sql FROM sqlite_master WHERE name='chunks_fts5';")
        sql = cur.fetchone()
        if sql:
            fts_sql = sql[0] or ""
            # Bad schema: content_rowid='id' where id is TEXT
            if "content_rowid='id'" in fts_sql:
                return False
        return True

    # ── FTS Search ──────────────────────────────────────────────────────────────

    def fts_search(self, query: str, top_k: Optional[int] = None) -> List[RetrievalResult]:
        """
        FTS search với SQLite FTS5 BM25 native.
        Sử dụng cấu trúc AND/OR thông minh dựa trên phân tích ngữ nghĩa để tránh gây nhiễu.
        Fallback sang LIKE search nếu FTS5 index chưa được tạo đúng.
        """
        k = top_k or self.top_k

        conn = self._get_conn()
        cur = conn.cursor()

        # Helper function to execute two-step FTS search to avoid SQLite slow JOIN behavior
        def _execute_fts_query(match_expr: str, limit: int) -> List[RetrievalResult]:
            try:
                cur.execute("""
                    SELECT rowid, -bm25(chunks_fts5) AS score
                    FROM chunks_fts5
                    WHERE chunks_fts5 MATCH ?
                    ORDER BY bm25(chunks_fts5)
                    LIMIT ?;
                """, (match_expr, limit))
                rows = cur.fetchall()
                if not rows:
                    return []

                rowids = [r[0] for r in rows]
                scores = {r[0]: float(r[1]) if r[1] is not None else 0.0 for r in rows}

                placeholders = ",".join("?" for _ in rowids)
                cur.execute(f"""
                    SELECT rowid, id, document_id, chunk_index, content
                    FROM document_chunks
                    WHERE rowid IN ({placeholders})
                """, rowids)
                details = cur.fetchall()

                results_list = []
                for row in details:
                    r_id, cid, doc_id, chunk_idx, content = row
                    score = scores.get(r_id, 0.0)
                    meta = _parse_meta_from_content(content or "")
                    results_list.append(RetrievalResult(
                        chunk_id=str(cid),
                        document_id=doc_id,
                        chunk_index=chunk_idx,
                        content=content or "",
                        doc_number=meta["document_number"],
                        title=meta["title"],
                        legal_type=meta["legal_type"],
                        score=score,
                        source="fts",
                        article_hint=extract_article_hint(content or ""),
                    ))
                results_list.sort(key=lambda x: x.score, reverse=True)
                return results_list
            except Exception as e:
                print(f"[local][FTS5] MATCH query error: {e}")
                return []

        if self._has_fts5_index():
            # Xây dựng câu truy vấn FTS cấu trúc AND/OR
            q_lower = query.lower()

            # Phân nhóm thực thể (Subjects) và Chủ đề/Hành động (Topics/Actions)
            subjects = {
                "incubator_coworking": {
                    "keywords": ["ươm tạo", "làm việc chung"],
                    "syns": ['"ươm tạo"', '"cơ sở ươm tạo"', '"khu làm việc chung"', '"làm việc chung"']
                },
                "sme": {
                    "keywords": ["doanh nghiệp nhỏ và vừa", "nhỏ và vừa", "sme", "siêu nhỏ"],
                    "syns": ['"doanh nghiệp nhỏ và vừa"', '"nhỏ và vừa"', '"doanh nghiệp nhỏ"', '"doanh nghiệp siêu nhỏ"']
                },
                "labor": {
                    "keywords": ["nhân viên", "lao động", "người lao động", "thử việc", "ký hợp đồng", "sa thải", "đuổi việc", "nghỉ việc", "lương"],
                    "syns": ['"người lao động"', '"lao động"', '"nhân viên"', '"hợp đồng lao động"']
                }
            }

            topics = {
                "tax_land": {
                    "keywords": ["thuế", "đất đai", "mặt bằng", "tiền thuê"],
                    "syns": ['"thuế"', '"đất đai"', '"mặt bằng"', '"thuê đất"', '"ưu đãi thuế"', '"tiền sử dụng đất"']
                },
                "bidding": {
                    "keywords": ["đấu thầu", "nhà thầu", "gói thầu"],
                    "syns": ['"đấu thầu"', '"nhà thầu"', '"lựa chọn nhà thầu"', '"gói thầu"', '"xếp hạng hồ sơ"']
                },
                "degrees_holding": {
                    "keywords": ["giữ", "bằng", "văn bằng", "chứng chỉ", "giấy tờ", "bằng cấp"],
                    "syns": ['"giữ"', '"thu giữ"', '"tạm giữ"', '"văn bằng"', '"chứng chỉ"', '"giấy tờ tùy thân"', '"bản chính"', '"bằng cấp"']
                }
            }

            active_subjects = []
            for s_name, s_info in subjects.items():
                if any(kw in q_lower for kw in s_info["keywords"]):
                    active_subjects.append(s_info["syns"])

            active_topics = []
            for t_name, t_info in topics.items():
                if any(kw in q_lower for kw in t_info["keywords"]):
                    active_topics.append(t_info["syns"])

            clauses = []
            if active_subjects:
                flat_subjects = []
                for s in active_subjects:
                    flat_subjects.extend(s)
                flat_subjects = list(dict.fromkeys(flat_subjects))
                clauses.append("(" + " OR ".join(flat_subjects) + ")")

            if active_topics:
                flat_topics = []
                for t in active_topics:
                    flat_topics.extend(t)
                flat_topics = list(dict.fromkeys(flat_topics))
                clauses.append("(" + " OR ".join(flat_topics) + ")")

            fts_expr = ""
            if clauses:
                fts_expr = " AND ".join(clauses)
                print(f"[local][FTS5] Structured query: {fts_expr}")
                results = _execute_fts_query(fts_expr, k)
                if results:
                    return results
                print(f"[local][FTS5] Structured query returned 0 results. Trying fallback OR...")

            # Fallback sang OR đơn giản của các từ khóa có nghĩa (loại bỏ stopword)
            GRAMMAR_STOPWORDS = {
                "và", "hoặc", "nhưng", "vì", "nên", "của", "các", "những", "là", "có",
                "trong", "tại", "để", "theo", "được", "bị", "cho", "ra", "vào", "lên",
                "xuống", "do", "từ", "đến", "bằng", "with", "với", "về", "như", "thì",
                "mà", "khi", "gì", "này", "đó", "kia", "nọ", "thế", "vậy", "nào", "ai",
                "đâu", "cái", "con", "chiếc", "nếu", "sẽ", "phải", "sao", "thế"
            }
            words = [w.lower().strip() for w in query.split()]
            terms = []
            for w in words:
                w = re.sub(r'[^\w\s\u00C0-\u024F\u1E00-\u1EFF]', '', w).strip()
                if w and len(w) >= 3 and w not in GRAMMAR_STOPWORDS:
                    terms.append(f'"{w}"')

            if terms:
                fallback_expr = " OR ".join(terms)
                print(f"[local][FTS5] Fallback OR query: {fallback_expr}")
                results = _execute_fts_query(fallback_expr, k)
                if results:
                    return results

            print("[local][FTS5] 0 results from FTS5. Trying LIKE fallback...")

        # LIKE fallback search (slower, for queries without diacritics)
        print("[local][FTS] Using LIKE fallback search...")
        keywords = [w for w in query.split() if len(w) >= 3][:5]
        if not keywords:
            return []

        like_pattern = f"%{keywords[0]}%"
        cur.execute("""
            SELECT id, document_id, chunk_index, content
            FROM document_chunks
            WHERE content LIKE ?
            LIMIT ?;
        """, (like_pattern, k * 3))
        rows = cur.fetchall()

        results = []
        for row in rows:
            cid, doc_id, chunk_idx, content = row[0], row[1], row[2], row[3] or ""
            score = float(sum(1 for kw in keywords if kw in content))
            if score == 0:
                continue
            meta = _parse_meta_from_content(content)
            results.append(RetrievalResult(
                chunk_id=str(cid),
                document_id=doc_id,
                chunk_index=chunk_idx,
                content=content,
                doc_number=meta["document_number"],
                title=meta["title"],
                legal_type=meta["legal_type"],
                score=score,
                source="fts",
                article_hint=extract_article_hint(content),
            ))
        results.sort(key=lambda x: -x.score)
        return results[:k]

    # ── FAISS Vector Search ───────────────────────────────────────────────────────

    def _get_faiss_index(self):
        """Lấy hoặc tự động build FAISS index từ SQLite database."""
        import faiss
        index_path = os.path.join(os.path.dirname(self.db_path), "local_chunks.index")

        if not hasattr(self, "_faiss_index") or self._faiss_index is None:
            if os.path.exists(index_path):
                print(f"[local] Loading FAISS index from {index_path}...", flush=True)
                t0 = time.time()
                self._faiss_index = faiss.read_index(index_path)
                print(f"[local] FAISS index loaded in {time.time()-t0:.2f}s", flush=True)
            else:
                print(f"[local] FAISS index not found. Building FAISS index from {self.db_path} (one-time setup)...", flush=True)
                t0 = time.time()
                conn = self._get_conn()
                cur = conn.cursor()
                cur.execute("SELECT rowid, embedding FROM document_chunks WHERE embedding IS NOT NULL;")
                rowids = []
                embeddings = []
                for rowid, emb_bytes in cur:
                    if not emb_bytes:
                        continue
                    emb = np.frombuffer(emb_bytes, dtype=np.float32)
                    rowids.append(rowid)
                    embeddings.append(emb)

                if not embeddings:
                    raise ValueError("No embeddings found in database to build FAISS index!")

                embeddings = np.array(embeddings, dtype=np.float32)
                rowids = np.array(rowids, dtype=np.int64)

                # Chuẩn hóa vector L2 để tìm kiếm Cosine Similarity bằng Inner Product
                norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
                norms[norms == 0] = 1.0
                embeddings = embeddings / norms

                dim = embeddings.shape[1]
                quantizer = faiss.IndexFlatIP(dim)
                index = faiss.IndexIDMap(quantizer)
                index.add_with_ids(embeddings, rowids)

                # Lưu index để các lần sau dùng ngay
                faiss.write_index(index, index_path)
                self._faiss_index = index
                print(f"[local] FAISS index built and saved in {time.time()-t0:.2f}s", flush=True)

        return self._faiss_index

    def vector_search(self, query: str, top_k: Optional[int] = None) -> List[RetrievalResult]:
        """Tìm kiếm tương đồng vector bằng FAISS index (tải và tìm cực nhanh)."""
        k = top_k or self.top_k

        query_vec = self.embed_query(query)
        query_norm = np.linalg.norm(query_vec)
        if query_norm == 0:
            return []

        # Chuẩn hóa query vector và định dạng cho FAISS
        query_vec = query_vec / query_norm
        query_vec = np.ascontiguousarray(query_vec.reshape(1, -1), dtype=np.float32)

        index = self._get_faiss_index()
        scores, indices = index.search(query_vec, k)
        scores = scores[0]
        indices = indices[0]

        valid_indices = [int(idx) for idx in indices if idx != -1]
        if not valid_indices:
            return []

        # Truy vấn SQLite để lấy thông tin các chunks tương ứng với rowid
        conn = self._get_conn()
        cur = conn.cursor()
        placeholders = ",".join("?" for _ in valid_indices)
        cur.execute(f"""
            SELECT rowid, id, document_id, chunk_index, content
            FROM document_chunks
            WHERE rowid IN ({placeholders});
        """, valid_indices)

        rows = cur.fetchall()
        row_map = {row["rowid"]: row for row in rows}

        results = []
        for idx, score in zip(indices, scores):
            idx = int(idx)
            if idx not in row_map:
                continue
            row = row_map[idx]
            cid = row["id"]
            doc_id = row["document_id"]
            chunk_idx = row["chunk_index"]
            content = row["content"] or ""

            meta = _parse_meta_from_content(content)
            results.append(RetrievalResult(
                chunk_id=str(cid),
                document_id=doc_id,
                chunk_index=chunk_idx,
                content=content,
                doc_number=meta["document_number"],
                title=meta["title"],
                legal_type=meta["legal_type"],
                score=float(score),
                source="vector",
                article_hint=extract_article_hint(content),
            ))
        return results

    # ── Hybrid (RRF) ────────────────────────────────────────────────────────────

    def hybrid_search(self, query: str, top_k: Optional[int] = None) -> List[RetrievalResult]:
        """Kết hợp FTS + vector bằng Reciprocal Rank Fusion."""
        k = top_k or self.top_k
        fetch_k = k * 3

        fts_results    = self.fts_search(query,    top_k=fetch_k)
        vector_results = self.vector_search(query, top_k=fetch_k)

        chunk_map: dict = {}
        rrf_scores: dict = {}

        def add_ranked(results: List[RetrievalResult], weight: float):
            for rank, r in enumerate(results, start=1):
                cid = r.chunk_id
                if cid not in chunk_map:
                    chunk_map[cid] = r
                    rrf_scores[cid] = 0.0
                rrf_scores[cid] += weight / (rank + self.rrf_k)

        add_ranked(fts_results,    self.fts_weight)
        add_ranked(vector_results, self.vector_weight)

        sorted_ids = sorted(rrf_scores, key=rrf_scores.get, reverse=True)
        results = []
        for cid in sorted_ids[:k]:
            r = chunk_map[cid]
            results.append(RetrievalResult(
                chunk_id=r.chunk_id,
                document_id=r.document_id,
                chunk_index=r.chunk_index,
                content=r.content,
                doc_number=r.doc_number,
                title=r.title,
                legal_type=r.legal_type,
                score=rrf_scores[cid],
                source="hybrid",
                article_hint=r.article_hint,
            ))
        return results

    def rerank(self, query: str, results: List[RetrievalResult], top_k: Optional[int] = None) -> List[RetrievalResult]:
        if not results:
            return []
    
        if self._reranker is None:
            import torch
            import time
            from sentence_transformers import CrossEncoder
    
            device = "cuda" if torch.cuda.is_available() else "cpu"
    
            print(
                f"[local] Loading reranker model BAAI/bge-reranker-v2-m3 on device '{device}'...",
                flush=True
            )
    
            t0 = time.time()
    
            automodel_args = {}
            if device == "cuda":
                automodel_args = {
                    "torch_dtype": torch.float16,
                    "low_cpu_mem_usage": True
                }
    
            self._reranker = CrossEncoder(
                "BAAI/bge-reranker-v2-m3",
                device=device,
                automodel_args=automodel_args
            )
    
            print(
                f"[local] Reranker model loaded in {time.time() - t0:.1f}s",
                flush=True
            )

        pairs = []
        for r in results:
            best_text = getattr(r, 'best_chunk_content', None) or r.content

            # Thêm metadata boosting để tăng độ chính xác (rất hiệu quả với văn bản pháp luật)
            article_str = f"Điều: {r.article_hint}\n" if r.article_hint else ""
            doc_text = f"Văn bản: {r.legal_type} {r.doc_number} - {r.title}\n{article_str}Nội dung:\n{best_text}"

            pairs.append([query, doc_text])

        t_rerank = time.time()
        scores = self._reranker.predict(pairs, show_progress_bar=False)
        print(f"[local] Reranking took {time.time() - t_rerank:.2f}s for {len(pairs)} pairs.")

        LEGAL_WEIGHT = {
            "Luật": 1.0,
            "Bộ luật": 1.0,
            "Nghị quyết": 0.95,
            "Nghị định": 0.85,
            "Thông tư": 0.75,
        }

        for r, score in zip(results, scores):
            rerank_score = float(score)
            # Normalize PhoRanker output to positive if it outputs logits
            if rerank_score < 0 or rerank_score > 1.0:
                import math
                rerank_score = 1 / (1 + math.exp(-max(min(rerank_score, 100), -100)))

            weight = LEGAL_WEIGHT.get(r.legal_type, 0.7)
            final_score = rerank_score * weight

            r.score = final_score
            r.source = f"rerank({r.source})"

        results.sort(key=lambda x: x.score, reverse=True)

        # Lọc MMR Diversity: max 2 chunk mỗi document
        filtered_results = []
        doc_count = {}
        for r in results:
            doc_id = r.document_id
            if doc_count.get(doc_id, 0) < 2:
                filtered_results.append(r)
                doc_count[doc_id] = doc_count.get(doc_id, 0) + 1

        if top_k:
            return filtered_results[:top_k]
        return filtered_results

    def are_chunks_duplicate(self, c1: RetrievalResult, c2: RetrievalResult) -> bool:
        """Kiểm tra xem hai chunk có trùng lặp nội dung đáng kể hay không."""
        if c1.document_id == c2.document_id and c1.chunk_index == c2.chunk_index:
            return True

        # Trích xuất phần nội dung thực tế để so sánh
        body1 = c1.content.split("| Nội dung:")[-1].strip()
        body2 = c2.content.split("| Nội dung:")[-1].strip()

        def normalize(text):
            text = text.lower()
            text = re.sub(r'[^\w\s]', '', text)
            return set(text.split())

        words1 = normalize(body1)
        words2 = normalize(body2)
        if not words1 or not words2:
            return False

        intersection = words1.intersection(words2)
        union = words1.union(words2)
        jaccard = len(intersection) / len(union)

        # Jaccard score cao => trùng lặp
        if jaccard > 0.8:
            return True

        # Nếu trùng Điều/Khoản luật và độ tương đồng tương đối cao => trùng lặp (ví dụ giữa Luật và VBHN)
        if c1.article_hint and c2.article_hint and c1.article_hint == c2.article_hint:
            if "điều" in c1.article_hint.lower() or "khoản" in c1.article_hint.lower():
                if jaccard > 0.6:
                    return True
        return False

    # ── Entry point ─────────────────────────────────────────────────────────────


    def expand_query(self, query: str) -> str:
        """Query Expansion bằng rule-based dictionary."""
        q_lower = query.lower()
        expanded_terms = []

        intents = {
            "hỗ trợ": ["ưu đãi", "chính sách hỗ trợ", "khuyến khích"],
            "ưu đãi": ["miễn", "giảm", "hỗ trợ"],
            "miễn": ["giảm", "không phải nộp", "ưu đãi"],
            "giảm": ["miễn", "ưu đãi thuế"],
            "đất đai": ["tiền thuê đất", "tiền sử dụng đất", "mặt bằng", "đất"],
            "thuế": ["thuế thu nhập doanh nghiệp", "thuế tndn", "miễn thuế", "giảm thuế"]
        }

        for k, v in intents.items():
            if k in q_lower:
                expanded_terms.extend(v)

        if expanded_terms:
            new_terms = [t for t in expanded_terms if t not in q_lower]
            if new_terms:
                return query + " " + " ".join(new_terms)
        return query

    def expand_to_parent_article(self, results: list) -> list:
        """Parent Retrieval: Lấy toàn bộ nội dung của Điều luật từ các chunk ban đầu."""
        if not results:
            return []

        target_articles = {}
        hit_indexes = {}
        for r in results:
            if not r.article_hint:
                continue
            doc_id = r.document_id
            if doc_id not in target_articles:
                target_articles[doc_id] = set()
            target_articles[doc_id].add(r.article_hint)

            # Ghi nhớ vị trí chunk_index đã hit
            key = (doc_id, r.article_hint)
            if key not in hit_indexes:
                hit_indexes[key] = set()
            hit_indexes[key].add(r.chunk_index)

        if not target_articles:
            return results

        doc_ids = list(target_articles.keys())
        conn = self._get_conn()
        cur = conn.cursor()

        placeholders = ",".join("?" for _ in doc_ids)
        cur.execute(f"""
            SELECT document_id, chunk_index, content
            FROM document_chunks
            WHERE document_id IN ({placeholders})
            ORDER BY document_id, chunk_index
        """, doc_ids)

        all_chunks = cur.fetchall()

        doc_chunks = {}
        for row in all_chunks:
            did = row["document_id"]
            if did not in doc_chunks:
                doc_chunks[did] = []
            doc_chunks[did].append(row)

        expanded_articles = {}

        for did, rows in doc_chunks.items():
            current_article = None
            for row in rows:
                content = row["content"] or ""
                # Rely on global extract_article_hint
                hint = extract_article_hint(content)
                if hint:
                    current_article = hint

                if current_article and current_article in target_articles[did]:
                    key = (did, current_article)

                    # Windowing limit: Chỉ gộp nếu chunk nằm gần vị trí hit (+/- 2 chunks)
                    # Chống phình to context đối với các Điều omnibus (VD: "sửa đổi, bổ sung một số điều")
                    hits = hit_indexes.get(key, set())
                    is_near_hit = False
                    for h in hits:
                        if abs(h - row["chunk_index"]) <= 2:
                            is_near_hit = True
                            break

                    if is_near_hit:
                        if key not in expanded_articles:
                            expanded_articles[key] = []
                        expanded_articles[key].append(content)

        final_results = []
        seen_keys = set()

        for r in results:
            if not r.article_hint:
                final_results.append(r)
                continue

            key = (r.document_id, r.article_hint)
            if key in seen_keys:
                continue
            seen_keys.add(key)

            if key in expanded_articles:
                # Lưu lại nội dung của best chunk (chunk ban đầu có điểm cao nhất) để dùng cho Rerank
                r.best_chunk_content = r.content

                full_content = "\n...\n".join(expanded_articles[key])
                r.content = full_content
                r.source += "_parent_expanded"
                final_results.append(r)
            else:
                final_results.append(r)

        return final_results

    def retrieve(
        self,
        query: str,
        mode: Literal["vector", "fts", "hybrid"] = "fts",
        top_k: Optional[int] = None,
        rerank: bool = True,
    ) -> List[RetrievalResult]:
        t0 = time.time()
        k = top_k or self.top_k

        # 1. Query Expansion (Intent Understanding)
        expanded_query = self.expand_query(query)

        # 2. Truy xuất danh sách ứng viên (candidate pool)
        if rerank:
            fetch_k = 50
        else:
            fetch_k = max(50, k * 4)

        if mode == "vector":
            results = self.vector_search(expanded_query, top_k=fetch_k)
        elif mode == "fts":
            results = self.fts_search(expanded_query, top_k=fetch_k)
        else:
            results = self.hybrid_search(expanded_query, top_k=fetch_k)

        # 3. Tiến hành lọc trùng lặp trước để loại bỏ nhiễu
        unique_results = []
        for r in results:
            is_dup = False
            for ur in unique_results:
                if self.are_chunks_duplicate(r, ur):
                     is_dup = True
                     break
            if not is_dup:
                unique_results.append(r)

        # 4. Parent Retrieval (Thay thế cho aggregate_chunks_into_articles)
        article_results = self.expand_to_parent_article(unique_results)

        # 5. Rerank và lọc
        if rerank:
            results = self.rerank(query, article_results, top_k=None)
        else:
            article_results.sort(key=lambda x: x.score, reverse=True)
            results = article_results

        # 4. Áp dụng Absolute & Relative Score Thresholds (Dynamic Thresholding)
        if results:
            RERANK_THRESHOLD = 0.15
            best_score = results[0].score

            relative_threshold_ratio = 0.5

            # Dynamic gap filtering: Nếu top 1 vượt trội hoàn toàn top 2
            if len(results) > 1:
                score_1 = results[0].score
                score_2 = results[1].score
                if (score_1 - score_2) > 0.2:
                    relative_threshold_ratio = 0.8

            filtered_by_score = []
            for r in results:
                # Giữ chunk nếu thỏa mãn ngưỡng tuyệt đối và ngưỡng tương đối động
                if r.score >= RERANK_THRESHOLD and r.score >= best_score * relative_threshold_ratio:
                    filtered_by_score.append(r)
            results = filtered_by_score

        # 5. Article-level Dedup (Giảm context thừa)
        seen_articles = set()
        deduped = []
        for r in results:
            article_id = r.format_relevant_article()
            if article_id in seen_articles:
                continue
            seen_articles.add(article_id)
            deduped.append(r)
        results = deduped

        # 6. Intent-aware (Query-aware) Filter
        try:
            import underthesea
            tags = underthesea.pos_tag(query)
            important_terms = [word.lower() for word, tag in tags if tag in ['N', 'Np', 'V', 'A', 'M'] and len(word) > 1]
            if not important_terms:
                important_terms = [w.lower() for w in query.split() if len(w) > 2]
        except Exception:
            important_terms = [w.lower() for w in query.split() if len(w) > 2]

        filtered_final = []
        for r in results:
            content_lower = r.content.lower()
            # Chunk phải chứa ít nhất một term quan trọng (intent keywords) từ câu hỏi
            if any(term in content_lower for term in important_terms):
                filtered_final.append(r)

        # Fallback nếu filter quá ngặt nghèo drop hết tất cả kết quả
        if not filtered_final and results:
            filtered_final = results

        results = filtered_final[:k]

        elapsed = time.time() - t0
        print(f"[local] mode={mode} | rerank={rerank} | {len(results)} results | {elapsed:.2f}s")
        return results

    @staticmethod
    def aggregate_by_document(results: List[RetrievalResult]) -> dict:
        """Nhóm kết quả theo document_id."""
        docs: dict = {}
        for r in results:
            if r.document_id not in docs:
                docs[r.document_id] = {
                    "doc_number": r.doc_number,
                    "title": r.title,
                    "legal_type": r.legal_type,
                    "chunks": [],
                    "max_score": 0.0,
                    "articles": set(),
                }
            docs[r.document_id]["chunks"].append(r)
            docs[r.document_id]["max_score"] = max(
                docs[r.document_id]["max_score"], r.score
            )
            if r.article_hint:
                docs[r.document_id]["articles"].add(r.article_hint)
        for doc in docs.values():
            try:
                doc["articles"] = sorted(
                    doc["articles"],
                    key=lambda x: int(re.search(r'\d+', x).group())
                )
            except Exception:
                doc["articles"] = sorted(doc["articles"])
        return docs


# ─── Quick CLI test ────────────────────────────────────────────────────────────

if __name__ == "__main__":
    if sys.platform == "win32":
        sys.stdout = io.TextIOWrapper(sys.stdout.buffer, encoding="utf-8", errors="replace")

    query = sys.argv[1] if len(sys.argv) > 1 else "doanh nghiệp nhỏ và vừa"
    mode  = sys.argv[2] if len(sys.argv) > 2 else "fts"
    top_k = int(sys.argv[3]) if len(sys.argv) > 3 else 5

    print(f"Query : {query}")
    print(f"Mode  : {mode}")
    print(f"Top-K : {top_k}")
    print("-" * 60)

    r = LocalRetriever(top_k=top_k)
    results = r.retrieve(query, mode=mode, top_k=top_k)
    r.close()

    for i, res in enumerate(results, 1):
        print(f"\n[{i}] {res.doc_number} | {res.legal_type}")
        print(f"    Tieu de : {res.title[:70]}")
        print(f"    Chunk#{res.chunk_index} | {res.article_hint or '-'} | score={res.score:.4f}")
        print(f"    Content : {res.content[:200]}...")

# ==========================================
# File: retrieval/pipeline_retriever.py
# ==========================================

from typing import List
import os
import sys

# Ensure project root is in path


class PipelineRetriever:
    """
    Step 3: BM25 (top 10) + Embedding (top 10)
    Step 4: Merge
    """
    def __init__(self, db_path=None, top_k_each=50, use_postgres=False):
        if db_path:
            self.retriever = LegalRetriever(db_path=db_path, use_postgres=use_postgres)
        else:
            self.retriever = LegalRetriever(use_postgres=use_postgres)
        self.top_k_each = top_k_each

    def retrieve_and_merge(self, query: str) -> List[RetrievalResult]:
        """Thực hiện song song FTS5 và Vector Search, sau đó merge bằng RRF."""
        # Step 3: Lấy top_k_each (10) cho mỗi phương pháp
        fts_results = self.retriever.fts_search(query, top_k=self.top_k_each)
        vector_results = self.retriever.vector_search(query, top_k=self.top_k_each)

        # Step 4: Merge (RRF)
        chunk_map = {}
        rrf_scores = {}

        # RRF k constant
        rrf_k = 60

        def add_ranked(results: List[RetrievalResult], weight: float):
            for rank, r in enumerate(results, start=1):
                cid = r.chunk_id
                if cid not in chunk_map:
                    chunk_map[cid] = r
                    rrf_scores[cid] = 0.0
                rrf_scores[cid] += weight / (rank + rrf_k)

        # Merge with equal weights
        add_ranked(fts_results, 0.5)
        add_ranked(vector_results, 0.5)

        # Lọc trùng lặp ngữ nghĩa (đã có hàm trong LocalRetriever)
        sorted_ids = sorted(rrf_scores, key=rrf_scores.get, reverse=True)
        merged_results = []
        for cid in sorted_ids:
            r = chunk_map[cid]
            # Tạo kết quả mới với score được cập nhật
            merged_results.append(RetrievalResult(
                chunk_id=r.chunk_id,
                document_id=r.document_id,
                chunk_index=r.chunk_index,
                content=r.content,
                doc_number=r.doc_number,
                title=r.title,
                legal_type=r.legal_type,
                score=rrf_scores[cid],
                source="merged_rrf",
                article_hint=r.article_hint,
            ))

        # Lọc trùng
        unique_results = []
        for r in merged_results:
            is_dup = False
            for ur in unique_results:
                if self.retriever.are_chunks_duplicate(r, ur):
                    is_dup = True
                    break
            if not is_dup:
                unique_results.append(r)

        # Gộp chunk thành Article
        grouped = {}
        for r in unique_results:
            key = (r.document_id, r.article_hint)
            if key not in grouped:
                grouped[key] = []
            grouped[key].append(r)

        article_results = []
        for key, chunks in grouped.items():
            chunks.sort(key=lambda x: x.chunk_index)
            base_chunk = chunks[0]
            if len(chunks) == 1 or not base_chunk.article_hint:
                article_results.append(base_chunk)
            else:
                combined_content = "\n...\n".join([c.content for c in chunks])
                max_score = max([c.score for c in chunks])

                new_res = RetrievalResult(
                    chunk_id=base_chunk.chunk_id,
                    document_id=base_chunk.document_id,
                    chunk_index=base_chunk.chunk_index,
                    content=combined_content,
                    doc_number=base_chunk.doc_number,
                    title=base_chunk.title,
                    legal_type=base_chunk.legal_type,
                    score=max_score,
                    source=base_chunk.source + "_aggregated",
                    article_hint=base_chunk.article_hint,
                )
                article_results.append(new_res)

        article_results.sort(key=lambda x: x.score, reverse=True)
        return article_results

# ==========================================
# File: retrieval/reranker.py
# ==========================================

import os
import sys
import time
from typing import List, Optional

# Ensure project root is in path


class PipelineReranker:
    """
    Step 5: Reranker (PhoRanker)
    Step 6: Top 5
    """
    def __init__(self, model_name="BAAI/bge-reranker-v2-m3", top_n=5): # Hoặc đường dẫn thư mục model BAAI bạn đã tải về (VD: "C:/path/to/bge-reranker-v2-m3")
        self.model_name = model_name
        self.top_n = top_n
        self._reranker = None
    def _lazy_load(self):
        if self._reranker is None:
            import torch
            from sentence_transformers import CrossEncoder
            device = "cuda" if torch.cuda.is_available() else "cpu"
            print(f"[pipeline] Loading reranker model {self.model_name} on '{device}'...", flush=True)
            t0 = time.time()
            automodel_args = {}
            if device == "cuda":
                automodel_args = {
                    "torch_dtype": torch.float16,
                    "low_cpu_mem_usage": True
                }
            self._reranker = CrossEncoder(self.model_name, device=device, automodel_args=automodel_args)
            print(f"[pipeline] Reranker loaded in {time.time()-t0:.1f}s", flush=True)

    def rerank_and_filter(self, query: str, results: List[RetrievalResult]) -> List[RetrievalResult]:
        """Rerank danh sách ứng viên và trả về Top N."""
        if not results:
            return []

        self._lazy_load()

        # Tokenize word level for Vietnamese query
        segmented_query = query
        try:
            import underthesea
            segmented_query = underthesea.word_tokenize(query, format="text")
        except Exception:
            pass

        pairs = []
        for r in results:
            # We use content as default if segmented_content is not available directly
            pairs.append([segmented_query, r.content])

        t_rerank = time.time()
        scores = self._reranker.predict(pairs, show_progress_bar=False)
        print(f"[pipeline] Reranked {len(pairs)} pairs in {time.time() - t_rerank:.2f}s.")

        LEGAL_BOOST = {
            "Luật": 0.3,
            "Nghị định": 0.2,
            "Thông tư": 0.1
        }

        for r, score in zip(results, scores):
            final_score = float(score)
            # Thêm điểm thưởng dựa trên loại văn bản pháp lý
            bonus = LEGAL_BOOST.get(r.legal_type, 0.0)
            if r.article_hint and "Điều" in r.article_hint:
                bonus += 0.1
            r.score = final_score + bonus
            r.source = f"rerank({r.source})"

        # Sort descending
        results.sort(key=lambda x: x.score, reverse=True)

        # Lọc max 2 chunk mỗi document
        filtered_results = []
        doc_count = {}
        for r in results:
            doc_id = r.document_id
            if doc_count.get(doc_id, 0) < 2:
                filtered_results.append(r)
                doc_count[doc_id] = doc_count.get(doc_id, 0) + 1

        # Step 6: Top 5
        return filtered_results[:self.top_n]

# ==========================================
# File: utils/context_compressor.py
# ==========================================

import re
from typing import List, Optional

import os
import sys

# Ensure project root is in path


class ContextCompressor:
    """
    Step 7: Context Compression
    Trích xuất đúng nội dung Điều luật (nếu có) và rút gọn context để tối ưu prompt cho LLM.
    """
    def __init__(self, max_length_per_doc=1500):
        self.max_length_per_doc = max_length_per_doc

    def extract_evidence(self, content: str, article_hint: Optional[str]) -> str:
        """Trích xuất duy nhất phần Điều luật liên quan từ chunk nội dung để tránh nhiễu."""
        header = ""
        lines = content.split('\n')
        for line in lines[:3]:
            if "Văn bản:" in line:
                header = line.strip() + "\n"
                break

        if not article_hint:
            return content.strip()

        # Hỗ trợ tìm kiếm cả số và chữ cái hậu tố (Ví dụ: 15, 15a, 15b)
        match_num = re.search(r'(\d+[a-zA-Z]?)', article_hint)
        if not match_num:
            return content.strip()

        art_num = match_num.group(1)

        pattern = re.compile(rf'^\s*Điều\s+{art_num}\b', re.MULTILINE | re.IGNORECASE)
        match = pattern.search(content)
        if not match:
            pattern_fallback = re.compile(rf'Điều\s+{art_num}\b', re.IGNORECASE)
            match = pattern_fallback.search(content)

        if not match:
            return content.strip()

        start_idx = match.start()

        next_pattern = re.compile(r'^\s*Điều\s+\d+\b', re.MULTILINE | re.IGNORECASE)
        matches_after = list(next_pattern.finditer(content[start_idx + len(match.group()):]))

        if matches_after:
            end_idx = start_idx + len(match.group()) + matches_after[0].start()
            extracted_text = content[start_idx:end_idx].strip()
        else:
            extracted_text = content[start_idx:].strip()

        return (header + extracted_text).strip()

    def compress(self, results: List[RetrievalResult]) -> str:
        """Nén danh sách Top 5 thành chuỗi context gọn gàng."""
        context_parts = []
        for idx, r in enumerate(results, start=1):
            extracted_body = self.extract_evidence(r.content, r.article_hint)

            # Tách bỏ header line "Văn bản: ..." nếu có
            lines = extracted_body.split('\n')
            body_lines = []
            for line in lines:
                if not line.strip().startswith("Văn bản:"):
                    body_lines.append(line)
            clean_body = "\n".join(body_lines).strip()

            # Truncate nếu quá dài
            if len(clean_body) > self.max_length_per_doc:
                clean_body = clean_body[:self.max_length_per_doc] + "...\n[Đã cắt bớt]"

            part = (
                f"[Căn cứ {idx}]\n"
                f"Độ liên quan: {r.score:.4f}\n"
                f"Văn bản: {r.legal_type} số {r.doc_number} {r.title}\n"
                f"Điều: {r.article_hint or 'Toàn bộ'}\n"
                f"Nội dung:\n{clean_body}"
            )
            context_parts.append(part)

        return "\n\n=========================================\n\n".join(context_parts)

# ==========================================
# File: utils/validator.py
# ==========================================

import re
from typing import List, Tuple

import os
import sys

# Ensure project root is in path


class PipelineValidator:
    """
    Step 9: Citation Validation
    Kiểm tra xem câu trả lời có lặp từ, hoặc trích dẫn các Điều/Số hiệu văn bản không có trong context hay không.
    """
    def __init__(self):
        pass

    def has_repetitive_loop(self, text: str) -> bool:
        """Kiểm tra xem câu trả lời có bị lặp từ/cụm từ vô hạn (loop) hay không."""
        lines = [line.strip() for line in text.split("\n") if line.strip()]
        for i in range(len(lines) - 2):
            if lines[i] == lines[i+1] == lines[i+2]:
                print(f"[validator] Phát hiện lặp dòng liên tiếp: '{lines[i]}'")
                return True

        for line in lines:
            words = line.split()
            if len(words) > 15:
                # Check sliding window of size 2 to 8
                for size in range(2, 9):
                    for start in range(len(words) - 3 * size):
                        w1 = words[start : start + size]
                        w2 = words[start + size : start + 2 * size]
                        w3 = words[start + 2 * size : start + 3 * size]
                        if w1 == w2 == w3:
                            print(f"[validator] Phát hiện lặp cụm từ vô hạn: {' '.join(w1)}")
                            return True
        return False

    def validate(self, answer: str, results: List[RetrievalResult]) -> Tuple[bool, str]:
        """
        Kiểm tra chi tiết trích dẫn của câu trả lời so với context.
        Trả về (is_valid, error_message)
        """
        if self.has_repetitive_loop(answer):
            return False, "câu trả lời bị lặp từ/cụm từ vô hạn (loop)"

        # Check Articles
        answer_articles = set(re.findall(r"Điều\s+(\d+)", answer, re.IGNORECASE))
        if answer_articles:
            context_articles = set()
            for r in results:
                if r.article_hint:
                    matches = re.findall(r"Điều\s+(\d+)", r.article_hint, re.IGNORECASE)
                    context_articles.update(matches)
                matches_content = re.findall(r"Điều\s+(\d+)", r.content, re.IGNORECASE)
                context_articles.update(matches_content)

            invalid_articles = answer_articles - context_articles
            if invalid_articles:
                return False, f"dẫn chiếu sai các Điều không có trong context: {invalid_articles}"

        # Check Documents
        answer_docs = set(re.findall(r"(\d+/\d+)", answer))
        if answer_docs:
            context_docs = set()
            for r in results:
                if r.doc_number:
                    matches = re.findall(r"(\d+/\d+)", r.doc_number)
                    context_docs.update(matches)
                matches_content = re.findall(r"(\d+/\d+)", r.content)
                context_docs.update(matches_content)

            invalid_docs = answer_docs - context_docs
            if invalid_docs:
                return False, f"dẫn chiếu sai các số hiệu Văn bản không có trong context: {invalid_docs}"

        return True, ""

# ==========================================
# File: main_pipeline.py
# ==========================================

import os
import sys

# Ensure project root is in path


class LegalRAGPipeline:
    """
    Toàn bộ RAG Pipeline chuẩn theo các bước:
    1: Question
    2: Query Rewrite
    3: BM25 (top 10) + Embedding (top 10)
    4: Merge
    5: Reranker
    6: Top 5
    7: Context Compression
    8: Qwen3-8B
    9: Citation Validation
    10: Final Answer
    """
    def __init__(self, use_llm_rewrite=False, db_path=None, llm_model_name="Qwen/Qwen3-8B-Instruct"):
        print("[pipeline] Khởi tạo các module...")
        self.generator = PipelineGenerator(model_name=llm_model_name)

        # Step 2
        self.rewriter = QueryRewriter(use_llm=use_llm_rewrite, llm_generator=self.generator)

        # Step 3 & 4
        self.retriever = PipelineRetriever(db_path=db_path, top_k_each=10)

        # Step 5 & 6
        self.reranker = PipelineReranker(top_n=5)

        # Step 7
        self.compressor = ContextCompressor()

        # Step 9
        self.validator = PipelineValidator()

        print("[pipeline] Khởi tạo hoàn tất.")

    def _generate_rule_based_answer(self, results) -> str:
        """Fallback: Tạo câu trả lời rule-based nếu LLM lỗi liên tục."""
        if not results:
            return "Không tìm thấy căn cứ pháp lý liên quan để trả lời câu hỏi này."

        answer_parts = ["1. Trả lời trực tiếp: Căn cứ vào các văn bản pháp luật hiện hành, dưới đây là thông tin trích xuất:"]
        analysis = []
        cơ_sở = []

        for idx, r in enumerate(results[:3], start=1):
            snippet = r.content.strip()
            if "Nội dung:" in snippet:
                snippet = snippet.split("Nội dung:", 1)[1].strip()
            if len(snippet) > 300:
                snippet = snippet[:297] + "..."
            analysis.append(f"- Căn cứ {idx} quy định: {snippet}")
            cơ_sở.append(f"- {r.article_hint or 'Quy định'} {r.legal_type} số {r.doc_number}")

        answer_parts.append("2. Phân tích chi tiết:\n" + "\n".join(analysis))
        answer_parts.append("3. Căn cứ pháp lý:\n" + "\n".join(cơ_sở))
        answer_parts.append("4. Hạn chế của dữ liệu (nếu có): Dựa trên trích xuất tự động.")
        return "\n\n".join(answer_parts)

    def run(self, question: str, max_retries: int = 2) -> dict:
        """
        Thực thi pipeline.
        Trả về dictionary chứa: query, rewritten_query, results (Top 5), context, answer, is_valid.
        """
        # Step 1: Question
        query = question.strip()
        if not query:
            return {"answer": "Câu hỏi trống."}

        # Step 2: Query Rewrite
        rewritten_query = self.rewriter.rewrite(query)

        # Step 3 & 4: Retrieval & Merge
        # Chúng ta dùng rewritten_query để search nhằm tăng độ bao phủ
        merged_results = self.retriever.retrieve_and_merge(rewritten_query)

        # Step 5 & 6: Reranker & Top 5
        # Reranker nên đánh giá độ liên quan với original_query để đảm bảo chính xác ngữ nghĩa gốc
        top5_results = self.reranker.rerank_and_filter(query, merged_results)

        # Step 7: Context Compression
        compressed_context = self.compressor.compress(top5_results)

        # Step 8, 9, 10: Generator, Validator, Final Answer
        final_answer = ""
        is_valid = False
        warning_msg = None

        for attempt in range(max_retries):
            # Step 8: Qwen3-8B
            answer = self.generator.generate_answer(
                query=query,
                context=compressed_context,
                warning_msg=warning_msg
            )

            # Step 9: Citation Validation
            valid, err_msg = self.validator.validate(answer, top5_results)
            if valid:
                final_answer = answer
                is_valid = True
                break
            else:
                print(f"[pipeline] Phát hiện lỗi sinh ({err_msg}) lần {attempt+1}. Đang sinh lại...")
                warning_msg = f"LƯU Ý LỚN: Ở lượt sinh trước, bạn đã mắc lỗi: {err_msg}. Hãy chú ý sửa lỗi này, không được lặp lại và không dẫn chiếu sai lệch."

        # Nếu vẫn không hợp lệ sau max_retries, fallback sang rule-based
        if not is_valid:
            print("[pipeline] Đã thử tối đa số lần, fallback sang Rule-based.")
            final_answer = self._generate_rule_based_answer(top5_results)

        # Step 10: Final Answer (trả về cùng metadata)
        return {
            "original_query": query,
            "rewritten_query": rewritten_query,
            "top5_results": top5_results,
            "compressed_context": compressed_context,
            "final_answer": final_answer,
            "is_valid": is_valid
        }

Query : -f
Mode  : /root/.local/share/jupyter/runtime/kernel-46b5d5b9-b0ea-450c-b1c1-d47a8e4b00dc.json
Top-K : 5
------------------------------------------------------------
[local][FTS5] 0 results from FTS5. Trying LIKE fallback...
[local][FTS] Using LIKE fallback search...
[local] Loading bkai-bi-encoder on device 'cuda'...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[local] Model loaded in 2.4s
[local] Loading FAISS index from /kaggle/working/R2AI/database/local_chunks.index...
[local] FAISS index loaded in 1.09s
[local] Loading reranker model BAAI/bge-reranker-v2-m3 on device 'cuda'...


The CrossEncoder `automodel_args` argument was renamed and is now deprecated. Please use `model_kwargs` instead.


config.json:   0%|          | 0.00/795 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[local] Model loaded in 2.5s
[local] Loading FAISS index from /kaggle/working/R2AI/database/local_chunks.index...
[local] FAISS index loaded in 1.13s
[local] Loading reranker model BAAI/bge-reranker-v2-m3 on device 'cuda'...


The CrossEncoder `automodel_args` argument was renamed and is now deprecated. Please use `model_kwargs` instead.


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

[local] Reranker model loaded in 4.9s
[local] Reranking took 0.75s for 38 pairs.
[local] mode=/root/.local/share/jupyter/runtime/kernel-46b5d5b9-b0ea-450c-b1c1-d47a8e4b00dc.json | rerank=True | 5 results | 9.60s

[1] 204/2015/TT-BTC | Thông tư
    Tieu de : quy định về áp dụng quản lý rủi ro trong quản lý thuế do Bộ trưởng Bộ 
    Chunk#9 | Điều 3 | score=0.4849
    Content : Văn bản: Thông tư 204/2015/TT-BTC quy định về áp dụng quản lý rủi ro trong quản lý thuế do Bộ trưởng Bộ Tài chính ban hành (204/2015/TT-BTC) | Điều 3. Giải thích từ ngữ (Phần 1) | Nội dung: Điều 3. Gi...

[2] 206/2012/TT-BTC | Thông tư
    Tieu de : hướng dẫn việc lập, quản lý, sử dụng Quỹ tập trung của Tập đoàn công n
    Chunk#1 | - | score=0.4563
    Content : Văn bản: Thông tư 206/2012/TT-BTC hướng dẫn việc lập, quản lý, sử dụng Quỹ tập trung của Tập đoàn công nghiệp Than-Khoáng sản Việt Nam do Bộ trưởng Bộ Tài chính ban hành (206/2012/TT-BTC) | Mục 0 (Phầ...

[3] 42/2019/TT-BLĐTBXH | Thông tư
    Tieu de : về

In [30]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder(
    "BAAI/bge-reranker-v2-m3",
    device="cuda"
)

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

In [35]:
import os
import json
import time
from concurrent.futures import ThreadPoolExecutor

# 5. Định nghĩa hàm chạy cho từng GPU (Chạy độc lập)
def process_chunk(gpu_id, questions_chunk, project_root):
    # Ép process này chỉ nhìn thấy 1 GPU duy nhất tương ứng với gpu_id (0 hoặc 1)
    os.environ["CUDA_VISIBLE_DEVICES"] = str(gpu_id)
    
    
    print(f"[GPU {gpu_id}] Initializing Pipeline...")
    # Khởi tạo Pipeline, truyền sẵn đường dẫn tới local database
    db_path = os.path.join(project_root, "database", "local_chunks.db")
    pipeline = LegalRAGPipeline(
        use_llm_rewrite=False,
        db_path=db_path,
        llm_model_name="Qwen/Qwen2.5-1.5B-Instruct",
    )
    
    # Override model name reranker to use local BAAI model
    pipeline.reranker.model_name = "BAAI/bge-reranker-v2-m3"
    print(f"[GPU {gpu_id}] Pre-loading all models into VRAM...")
    # Load LLM Generator
    pipeline.generator._lazy_load()
    # Load Reranker (PhoRanker)
    pipeline.reranker._lazy_load()
    # Load Embedder (vietnamese-bi-encoder)
    pipeline.retriever.retriever._get_model()
    # Load FAISS Index
    pipeline.retriever.retriever._get_faiss_index()
    print(f"[GPU {gpu_id}] All models loaded successfully!")
    
    results = []
    for i, q in enumerate(questions_chunk):
        qid = q.get("id")
        question = q.get("question", "")
        print(f"[GPU {gpu_id}] [{i+1}/{len(questions_chunk)}] Processing: {question[:50]}...")
        
        t0 = time.time()
        try:
            out = pipeline.run(question)
            
            relevant_docs = []
            docs_seen = set()
            for r in out.get("top5_results", []):
                doc_str = r.format_relevant_doc()
                if doc_str not in docs_seen:
                    docs_seen.add(doc_str)
                    relevant_docs.append(doc_str)

            relevant_articles = []
            articles_seen = set()
            for r in out.get("top5_results", []):
                if r.article_hint:
                    art_str = r.format_relevant_article()
                    if art_str not in articles_seen:
                        articles_seen.add(art_str)
                        relevant_articles.append(art_str)

            results.append({
                "id": qid,
                "question": question,
                "answer": out.get("final_answer", ""),
                "relevant_docs": relevant_docs,
                "relevant_articles": relevant_articles
            })
            print(f"[GPU {gpu_id}] Done in {time.time() - t0:.2f}s")
            
        except Exception as e:
            print(f"[GPU {gpu_id}] ERROR processing ID {qid}: {e}")
            results.append({
                "id": qid,
                "question": question,
                "answer": "",
                "relevant_docs": [],
                "relevant_articles": []
            })
            
    return results

In [ ]:
# 6. Đọc dữ liệu và chia việc ra 2 GPUs
from concurrent.futures import ThreadPoolExecutor
data_path = os.path.join(PROJECT_ROOT, "R2AIStage1DATA.json")
with open(data_path, "r", encoding="utf-8") as f:
    test_questions = json.load(f)

print(f"Loaded {len(test_questions)} questions.")

# Chia danh sách câu hỏi làm 2 phần bằng nhau
mid = len(test_questions) // 2
chunks = [
    (0, test_questions[:mid], PROJECT_ROOT),   # Process 1 chạy trên GPU 0
    (1, test_questions[mid:], PROJECT_ROOT)    # Process 2 chạy trên GPU 1
]

start_time = time.time()
final_results = []

# Chạy song song 2 process (Mỗi process khởi tạo 1 pipeline trên 1 GPU riêng biệt)
with ThreadPoolExecutor(max_workers=4) as executor:
    futures = [executor.submit(process_chunk, *args) for args in chunks]
    for future in futures:
        final_results.extend(future.result())

# Sắp xếp lại theo ID ban đầu (vì chạy song song thứ tự có thể lộn xộn)
final_results.sort(key=lambda x: x['id'])

total_time = time.time() - start_time
print(f"\nCompleted all {len(test_questions)} questions in {total_time:.2f}s with 2 GPUs.")

Loaded 2000 questions.
[GPU 0] Initializing Pipeline...
[pipeline] Khởi tạo các module...
[pipeline] Khởi tạo hoàn tất.
[GPU 0] Pre-loading all models into VRAM...
[generator] Loading model 'Qwen/Qwen2.5-1.5B-Instruct' (Offline=False)...
[GPU 1] Initializing Pipeline...
[pipeline] Khởi tạo các module...
[pipeline] Khởi tạo hoàn tất.
[GPU 1] Pre-loading all models into VRAM...
[generator] Loading model 'Qwen/Qwen2.5-1.5B-Instruct' (Offline=False)...
[generator] Device=cuda, Dtype=torch.bfloat16...
[generator] Device=cuda, Dtype=torch.bfloat16...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

[generator] Model loaded in 4.3s
[pipeline] Loading reranker model BAAI/bge-reranker-v2-m3 on 'cuda'...


The CrossEncoder `automodel_args` argument was renamed and is now deprecated. Please use `model_kwargs` instead.


...

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

[pipeline] Reranker loaded in 5.3s
[local] Loading bkai-bi-encoder on device 'cuda'...
[pipeline] Reranker loaded in 7.9s
[local] Loading bkai-bi-encoder on device 'cuda'...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[local] Model loaded in 4.6s
[local] Loading FAISS index from /kaggle/working/R2AI/database/local_chunks.index...
[local] Model loaded in 2.6s
[local] Loading FAISS index from /kaggle/working/R2AI/database/local_chunks.index...
[local] FAISS index loaded in 1.12s
[GPU 1] All models loaded successfully!
[GPU 1] [1/1000] Processing: Khi công ty chuyển đổi loại hình doanh nghiệp, để ...
[local][FTS5] Fallback OR query: "công" OR "chuyển" OR "đổi" OR "loại" OR "hình" OR "doanh" OR "nghiệp" OR "đảm" OR "bảo" OR "tài" OR "sản" OR "toàn" OR "báo" OR "cáo" OR "tài" OR "chính" OR "trung" OR "thực" OR "trước" OR "bàn" OR "giao" OR "công" OR "cần" OR "thực" OR "hiện" OR "công" OR "việc" OR "toán" OR "đoàn" OR "kiểm" OR "tra" OR "toán" OR "thời" OR "điểm" OR "quyền" OR "hạn"
[local] FAISS index loaded in 1.11s
[GPU 0] All models loaded successfully!
[GPU 0] [1/1000] Processing: Các cơ sở ươm tạo và khu làm việc chung được hưởng...
[local][FTS5] Structured query: ("ươm tạo" OR "cơ sở ươm tạo" OR "k

In [23]:
import torch

print("GPU count:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    print(i, torch.cuda.get_device_name(i))

GPU count: 2
0 Tesla T4
1 Tesla T4


In [ ]:
# 7. Lưu file kết quả xuống thư mục Working
output_path = "/kaggle/working/submission.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(final_results, f, ensure_ascii=False, indent=2)

print(f"Results saved to {output_path}")
print("Bạn có thể download file submission.json từ cột bên phải của Kaggle!")